# Setup

In [ ]:
pip install rpy2

In [ ]:
%load_ext rpy2.ipython

## Install R Packages

In [ ]:
%%R

install.packages("gtsummary")

In [ ]:
%%R
install.packages("survival")
install.packages("survminer")
install.packages("lubridate")
install.packages("anytime")
install.packages("DataExplorer")
install.packages("patchwork")
install.packages("MatchIt")

In [ ]:
%%R

install.packages("caret")

In [ ]:
%%R

install.packages("gt")
install.packages("gtExtras")

In [ ]:
%%R

install.packages("pROC")

In [ ]:
%%R

install.packages("grid")
install.packages("gridExtra")

## Load R + Python Packages

In [ ]:
%%R
library(tidyverse)
library(survival)
library(survminer)
library(anytime)
library(lubridate)
library(DataExplorer)
library(patchwork)
library(pROC)
library(grid)
library(gridExtra)
library(gt)
library(gtExtras)
library(caret)
library(MatchIt)
library(gtsummary)

In [ ]:
import os
import numpy as np
import pandas as pd
import pandas_profiling
import subprocess
import plotnine
import matplotlib.pyplot as plt
#import lifelines
#from lifelines import KaplanMeierFitter
from plotnine import *  # Provides a ggplot-like interface to matplotlib.
from IPython.display import display
from datetime import datetime

## Plot setup.
theme_set(theme_bw(base_size = 11)) # Default theme for plots.

def get_boxplot_fun_data(df):
  """Returns a data frame with a y position and a label, for use annotating ggplot boxplots.

  Args:
    d: A data frame.
  Returns:
    A data frame with column y as max and column label as length.
  """
  d = {'y': max(df), 'label': f'N = {len(df)}'}
  return(pd.DataFrame(data=d, index=[0]))

# NOTE: if you get any errors from this cell, restart your kernel and run it again.


# Import Kidney Stone Data

In [ ]:
import pandas
import os

# This query represents dataset "Kidney stones - symptomatic occurrence >6 months" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_43325964_condition_sql = """
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (201690, 201916, 4066163)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1) 
                OR  condition_source_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (1571487, 35209253, 35209281, 35209282, 35209283, 35209284, 35209290, 44825543, 44826732, 44831593, 44833664, 44837195)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 0 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            person_id IN (SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                concept_id IN(SELECT
                                    DISTINCT c.concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id       
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                    WHERE
                                        concept_id IN (4237124, 4121225, 4087889, 4301429, 2721096, 4341377, 4170611, 4171381)       
                                        AND full_text LIKE '%_rank1]%'      ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) 
                                AND is_standard = 1 
                            UNION
                            DISTINCT SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                concept_id IN(SELECT
                                    DISTINCT c.concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id       
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                    WHERE
                                        concept_id IN (2109816, 2109804, 2109554, 2109556, 2109803, 2774224, 2109687, 2832813, 2109557, 2109563, 2109553, 2824569, 2109634, 2774220, 2109635, 2008219, 2008218, 2721096, 2819509, 2109626, 2109817, 2109641, 2003682, 2109558, 44816428)       
                                        AND full_text LIKE '%_rank1]%'      ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 0 
                                    AND is_selectable = 1) 
                                AND is_standard = 0 )) criteria 
                        UNION
                        DISTINCT SELECT
                            criteria.person_id 
                        FROM
                            (SELECT
                                DISTINCT person_id, entry_date, concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                person_id IN (SELECT
                                    person_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    concept_id IN(SELECT
                                        DISTINCT c.concept_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                    JOIN
                                        (SELECT
                                            CAST(cr.id as string) AS id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                        WHERE
                                            concept_id IN (201690, 4066163, 201916)       
                                            AND full_text LIKE '%_rank1]%'      ) a 
                                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                            OR c.path LIKE CONCAT('%.', a.id) 
                                            OR c.path LIKE CONCAT(a.id, '.%') 
                                            OR c.path = a.id) 
                                    WHERE
                                        is_standard = 1 
                                        AND is_selectable = 1) 
                                    AND is_standard = 1 
                                UNION
                                DISTINCT SELECT
                                    person_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    concept_id IN(SELECT
                                        DISTINCT c.concept_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                    JOIN
                                        (SELECT
                                            CAST(cr.id as string) AS id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                        WHERE
                                            concept_id IN (1571487, 44831593, 44837195, 35209290, 35209253)       
                                            AND full_text LIKE '%_rank1]%'      ) a 
                                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                            OR c.path LIKE CONCAT('%.', a.id) 
                                            OR c.path LIKE CONCAT(a.id, '.%') 
                                            OR c.path = a.id) 
                                    WHERE
                                        is_standard = 0 
                                        AND is_selectable = 1) 
                                    AND is_standard = 0 )) criteria ) 
                                AND cb_search_person.person_id IN (SELECT
                                    person_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                                WHERE
                                    has_whole_genome_variant = 1 ) )
                            )
                        ) c_occurrence 
                    LEFT JOIN
                        `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_standard_concept 
                            ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
                    LEFT JOIN
                        `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_type 
                            ON c_occurrence.condition_type_concept_id = c_type.concept_id 
                    LEFT JOIN
                        `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                            ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
                    LEFT JOIN
                        `""" + os.environ["WORKSPACE_CDR"] + """.concept` visit 
                            ON v.visit_concept_id = visit.concept_id 
                    LEFT JOIN
                        `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_source_concept 
                            ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
                    LEFT JOIN
                        `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_status 
                            ON c_occurrence.condition_status_concept_id = c_status.concept_id"""

symptomatic_occurrence_diagnostic_codes = pandas.read_gbq(
    dataset_43325964_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

symptomatic_occurrence_diagnostic_codes.head(10)

In [ ]:
import pandas
import os

# This query represents dataset "Kidney stones - symptomatic occurrence >6 months" for domain "procedure" and was generated for All of Us Controlled Tier Dataset v8
dataset_43325964_procedure_sql = """
    SELECT
        procedure.person_id,
        procedure.procedure_concept_id,
        p_standard_concept.concept_name as standard_concept_name,
        p_standard_concept.concept_code as standard_concept_code,
        p_standard_concept.vocabulary_id as standard_vocabulary,
        procedure.procedure_datetime,
        procedure.procedure_type_concept_id,
        p_type.concept_name as procedure_type_concept_name,
        procedure.modifier_concept_id,
        p_modifier.concept_name as modifier_concept_name,
        procedure.quantity,
        procedure.visit_occurrence_id,
        p_visit.concept_name as visit_occurrence_concept_name,
        procedure.procedure_source_value,
        procedure.procedure_source_concept_id,
        p_source_concept.concept_name as source_concept_name,
        p_source_concept.concept_code as source_concept_code,
        p_source_concept.vocabulary_id as source_vocabulary,
        procedure.modifier_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.procedure_occurrence` procedure 
        WHERE
            (
                procedure_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (2721096, 4087889, 4121225, 4170611, 4171381, 4237124, 4301429, 4341377)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1) 
                OR  procedure_source_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (2003682, 2008219, 2008220, 2008221, 2109553, 2109556, 2109557, 2109558, 2109563, 2109626, 2109634, 2109635, 2109641, 2109687, 2109803, 2109804, 2109816, 2109817, 2721096, 2774214, 2774218, 2774219, 2774220, 2774224, 44816428)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 0 
                    AND is_selectable = 1)
            )  
            AND (
                procedure.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            person_id IN (SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                concept_id IN(SELECT
                                    DISTINCT c.concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id       
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                    WHERE
                                        concept_id IN (4237124, 4121225, 4087889, 4301429, 2721096, 4341377, 4170611, 4171381)       
                                        AND full_text LIKE '%_rank1]%'      ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) 
                                AND is_standard = 1 
                            UNION
                            DISTINCT SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                concept_id IN(SELECT
                                    DISTINCT c.concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id       
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                    WHERE
                                        concept_id IN (2109816, 2109804, 2109554, 2109556, 2109803, 2774224, 2109687, 2832813, 2109557, 2109563, 2109553, 2824569, 2109634, 2774220, 2109635, 2008219, 2008218, 2721096, 2819509, 2109626, 2109817, 2109641, 2003682, 2109558, 44816428)       
                                        AND full_text LIKE '%_rank1]%'      ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 0 
                                    AND is_selectable = 1) 
                                AND is_standard = 0 )) criteria 
                        UNION
                        DISTINCT SELECT
                            criteria.person_id 
                        FROM
                            (SELECT
                                DISTINCT person_id, entry_date, concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                person_id IN (SELECT
                                    person_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    concept_id IN(SELECT
                                        DISTINCT c.concept_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                    JOIN
                                        (SELECT
                                            CAST(cr.id as string) AS id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                        WHERE
                                            concept_id IN (201690, 4066163, 201916)       
                                            AND full_text LIKE '%_rank1]%'      ) a 
                                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                            OR c.path LIKE CONCAT('%.', a.id) 
                                            OR c.path LIKE CONCAT(a.id, '.%') 
                                            OR c.path = a.id) 
                                    WHERE
                                        is_standard = 1 
                                        AND is_selectable = 1) 
                                    AND is_standard = 1 
                                UNION
                                DISTINCT SELECT
                                    person_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    concept_id IN(SELECT
                                        DISTINCT c.concept_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                    JOIN
                                        (SELECT
                                            CAST(cr.id as string) AS id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                        WHERE
                                            concept_id IN (1571487, 44831593, 44837195, 35209290, 35209253)       
                                            AND full_text LIKE '%_rank1]%'      ) a 
                                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                            OR c.path LIKE CONCAT('%.', a.id) 
                                            OR c.path LIKE CONCAT(a.id, '.%') 
                                            OR c.path = a.id) 
                                    WHERE
                                        is_standard = 0 
                                        AND is_selectable = 1) 
                                    AND is_standard = 0 )) criteria ) 
                                AND cb_search_person.person_id IN (SELECT
                                    person_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                                WHERE
                                    has_whole_genome_variant = 1 ) )
                            )
                        ) procedure 
                    LEFT JOIN
                        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_standard_concept 
                            ON procedure.procedure_concept_id = p_standard_concept.concept_id 
                    LEFT JOIN
                        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_type 
                            ON procedure.procedure_type_concept_id = p_type.concept_id 
                    LEFT JOIN
                        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_modifier 
                            ON procedure.modifier_concept_id = p_modifier.concept_id 
                    LEFT JOIN
                        `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                            ON procedure.visit_occurrence_id = v.visit_occurrence_id 
                    LEFT JOIN
                        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_visit 
                            ON v.visit_concept_id = p_visit.concept_id 
                    LEFT JOIN
                        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_source_concept 
                            ON procedure.procedure_source_concept_id = p_source_concept.concept_id"""

symptomatic_occurrence_procedure_codes = pandas.read_gbq(
    dataset_43325964_procedure_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

symptomatic_occurrence_procedure_codes.head(10)

# Import and Sort Final date dataset

## Original Final Dates person data

In [ ]:
import pandas
import os

# This query represents dataset "Final Dates" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_78141828_person_sql = """
    SELECT
        person.person_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (40480457, 4042140, 4011630, 4274025, 4168335, 4094294, 4021667)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria 
            UNION
            DISTINCT SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (2617204, 4029205, 704057, 4180627, 2617208, 1314331, 2720580, 4237321, 4180941, 40295754, 40217302, 4302541, 4132623, 45890623, 4028908, 4072499, 2721530, 4237320, 4176642)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria ) )"""

overall_person_df = pandas.read_gbq(
    dataset_78141828_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

overall_person_df.head(5)

## New Final Dates person data - filter for ethnicity

In [ ]:
import pandas
import os

# This query represents dataset "AoU Entire cohort - demographics + final dates" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_28927342_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (40480457, 4042140, 4011630, 4274025, 4168335, 4094294, 4021667)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria 
            UNION
            DISTINCT SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (2617204, 4029205, 704057, 4180627, 2617208, 1314331, 2720580, 4237321, 4180941, 40295754, 40217302, 4302541, 4132623, 45890623, 4028908, 4072499, 2721530, 4237320, 4176642)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria ) )"""

overall_person_df_ethnicity = pandas.read_gbq(
    dataset_28927342_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

overall_person_df_ethnicity.head(5)

In [ ]:
ethnicity_factors = overall_person_df_ethnicity['race'].unique()
print(f'Different factors in the ethnicity column: {ethnicity_factors}')

total_count = overall_person_df_ethnicity.shape[0]

# Count participants identified as White
white_count = overall_person_df_ethnicity[overall_person_df_ethnicity['race'] == 'White'].shape[0]

# Calculate proportion
white_proportion = white_count / total_count

# Display the result
print(f'Proportion of White participants: {white_proportion:.2%}')

In [ ]:
import pandas
import os

# This query represents dataset "Final Dates" for domain "procedure" and was generated for All of Us Controlled Tier Dataset v8
dataset_88903298_procedure_sql = """
    SELECT
        procedure.person_id,
        procedure.procedure_concept_id,
        p_standard_concept.concept_name as standard_concept_name,
        p_standard_concept.concept_code as standard_concept_code,
        p_standard_concept.vocabulary_id as standard_vocabulary,
        procedure.procedure_datetime,
        procedure.procedure_type_concept_id,
        p_type.concept_name as procedure_type_concept_name,
        procedure.modifier_concept_id,
        p_modifier.concept_name as modifier_concept_name,
        procedure.quantity,
        procedure.visit_occurrence_id,
        p_visit.concept_name as visit_occurrence_concept_name,
        procedure.procedure_source_value,
        procedure.procedure_source_concept_id,
        p_source_concept.concept_name as source_concept_name,
        p_source_concept.concept_code as source_concept_code,
        p_source_concept.vocabulary_id as source_vocabulary,
        procedure.modifier_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.procedure_occurrence` procedure 
        WHERE
            (
                procedure_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (1314331, 2617204, 2617208, 2617223, 2720580, 2721530, 36675054, 36714147, 37204303, 37204304, 4000738, 4001217, 4002365, 4002733, 4003041, 4004516, 4010109, 4012185, 4012907, 4013636, 4016388, 4017459, 40217302, 4024612, 4027403, 4028902, 4028908, 4028909, 4029205, 40295754, 4029698, 4030028, 4030654, 4030660, 4031967, 4032237, 4032251, 4032254, 4032257, 4036803, 4037672, 4039376, 4040390, 4040391, 4040392, 4040393, 4040395, 4040396, 4040400, 4040551, 4040556, 4040559, 4040561, 4040563, 4040721, 4041261, 4041263, 4041982, 4042150, 4042330, 4042486, 4042493, 4042504, 4042516, 4042641, 4042642, 4042645, 4042646, 4042650, 4042660, 4042673, 4042677, 4042815, 4043017, 4043018, 4043019, 4043022, 4043023, 4043025, 4043028, 4043161, 4043176, 4043178, 4043179, 4043182, 4043197, 4043334, 4043336, 4044012, 4044875, 4044887, 4044900, 4045893, 4045900, 4047347, 40480674, 40480925, 40481382, 40481384, 40481878, 40482428, 4048375, 40487842, 4052492, 4063521,
 4064522, 4065062, 4067285, 4072499, 4073050, 4073623, 4074287, 4075084, 4075966, 4077953, 4078442, 4078460, 4079711, 4088217, 4093436, 4100357, 4104795, 4105895, 4108616, 4109665, 4111522, 4111529, 4111534, 4117767, 4118601, 4121315, 4124406, 4125501, 4128030, 4128032, 4129747, 4132623, 4132648, 4132855, 4133169, 4133312, 4134148, 4134423, 4134598, 4135174, 4138127, 4139262, 4141759, 4144375, 4145308, 4147894, 4155790, 4155793, 4158570, 4159949, 4160360, 4160550, 4162054, 4163951, 4164278, 4168709, 4172515, 4172654, 4176642, 4176720, 4177094, 4178367, 4179713, 4180004, 4180627, 4180642, 4180653, 4180938, 4180941, 4181192, 4181193, 4181322, 4181638, 4181966, 4185115, 4185623, 4189705, 4190496, 4190759, 4194672, 4195757, 4196582, 4196958, 4201944, 4202517, 4202832, 4208079, 4211641, 4213297, 4214115, 4216752, 4221994, 4230374, 4230911, 4231758, 4233946, 4237320, 4237321, 4239679, 4240345, 4241075, 4243910, 4247168, 4248675, 4249893, 4250598, 42536500, 4254477, 4254480, 4258374, 4261206,
 4263508, 4264477, 4266234, 4268628, 4270484, 4279903, 4287856, 4293032, 4293902, 4297090, 4297249, 4299523, 4299602, 4300244, 4300757, 4301351, 4302532, 4302541, 4304358, 4306074, 4306780, 4310912, 4311405, 4313306, 4313889, 4314450, 4324693, 4326426, 43531071, 43531072, 45773385, 45890623, 46272569, 704057)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                procedure.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (40480457, 4042140, 4011630, 4274025, 4168335, 4094294, 4021667)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (2617204, 4029205, 704057, 4180627, 2617208, 1314331, 2720580, 4237321, 4180941, 40295754, 40217302, 4302541, 4132623, 45890623, 4028908, 4072499, 2721530, 4237320, 4176642)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria ) )
                )
            ) procedure 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_standard_concept 
                ON procedure.procedure_concept_id = p_standard_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_type 
                ON procedure.procedure_type_concept_id = p_type.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_modifier 
                ON procedure.modifier_concept_id = p_modifier.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                ON procedure.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_visit 
                ON v.visit_concept_id = p_visit.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_source_concept 
                ON procedure.procedure_source_concept_id = p_source_concept.concept_id"""


overall_procedures_df = pandas.read_gbq(
    dataset_88903298_procedure_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

overall_procedures_df.head(5)

In [ ]:
import pandas
import os

# This query represents dataset "Final Dates" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_88903298_condition_sql = """
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (133468, 134057, 134736, 135930, 138239, 138525, 140190, 141960, 192438, 193460, 194133, 200174, 200219, 200452, 201618, 201891, 253549, 254068, 254761, 255919, 31610, 316866, 31821, 318800, 320128, 320136, 321588, 37018424, 37018713, 37203927, 37206233, 372887, 37311677, 37311678, 373499, 375252, 376106, 376208, 376337, 4000609, 4000610, 4002898, 4002905, 4006969, 4009890, 4011630, 4020346, 4021667, 4021780, 4021915, 4022201, 4022922, 4022923, 4022924, 4023995, 4024000, 4024013, 4024561, 4024566, 4024567, 4025215, 4027384, 4027553, 4028071, 4028076, 4028387, 4031814, 4038502, 4038677, 4038678, 4041283, 4041284, 4041285, 4041436, 4042056, 4042140, 4042141, 4042142, 4042503, 4042505, 4042835, 4042836, 4042837, 4043346, 4043371, 4043671, 4047779, 40480457, 40481517, 40481841, 40484102, 40484533, 40484935, 4051221, 4054501, 4071689, 4080992, 4081176, 4083787, 4086181, 4090426, 4090614, 4090615, 4091213, 4091363, 4091532, 4093227, 4093228, 4093991,
 4094294, 4095793, 4096864, 4100932, 4101212, 4101343, 4101796, 4102111, 4103183, 4103320, 4103352, 4104157, 4113563, 4113999, 4115259, 4115386, 4115390, 4116811, 4117779, 4117930, 4132555, 4132926, 4134294, 4134440, 4150129, 4160062, 4162282, 4168335, 4170143, 4170226, 4170962, 4171379, 4175154, 4178545, 4178680, 4178818, 4179141, 4179167, 4179871, 4179873, 4180154, 4180169, 4180170, 4180628, 4181063, 4181187, 4182161, 4182165, 4182633, 4184252, 4185503, 4185640, 4190185, 4197094, 4198525, 4199402, 4200532, 4201745, 4206591, 4208786, 4212577, 4213101, 4221108, 4227253, 4244662, 4247371, 42538830, 4260918, 4266186, 4267789, 4269314, 4272867, 4274025, 4276569, 4276572, 4277352, 4293175, 4297887, 4301699, 4302537, 4304916, 4305027, 4308811, 4309188, 4316083, 4317258, 4318379, 432250, 432586, 432795, 432867, 4329041, 433128, 433736, 4338120, 4339410, 4339468, 4344497, 43530815, 43530877, 43531053, 43531054, 43531056, 43531057, 43531058, 43531059, 43531060, 435506, 435524, 436096, 436670,
 437515, 438112, 440029, 440142, 440371, 440383, 440921, 441542, 442077, 443343, 443723, 443783, 443784, 443883, 444089, 444100, 444108, 444112, 444363, 44782620, 44783587, 44784102, 73553, 75865, 75909, 77074, 77670, 77960, 79106, 80180)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (40480457, 4042140, 4011630, 4274025, 4168335, 4094294, 4021667)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (2617204, 4029205, 704057, 4180627, 2617208, 1314331, 2720580, 4237321, 4180941, 40295754, 40217302, 4302541, 4132623, 45890623, 4028908, 4072499, 2721530, 4237320, 4176642)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria ) )
                )
            ) c_occurrence 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` visit 
                ON v.visit_concept_id = visit.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_source_concept 
                ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_status 
                ON c_occurrence.condition_status_concept_id = c_status.concept_id"""

overall_condition_df = pandas.read_gbq(
    dataset_88903298_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

overall_condition_df.head(5)

In [ ]:
overall_person_df.head()

In [ ]:
import pandas as pd

# Select and rename from symptomatic procedure codes
symp_proc_selected = symptomatic_occurrence_procedure_codes[['person_id', 'procedure_datetime']].rename(
    columns={'procedure_datetime': 'date_time'}
)

overall_procedure_selected = overall_procedures_df[['person_id', 'procedure_datetime']].rename(
    columns={'procedure_datetime': 'date_time'}
)

# Select and rename from symptomatic diagnostic codes
symp_diag_selected = symptomatic_occurrence_diagnostic_codes[['person_id', 'condition_start_datetime']].rename(
    columns={'condition_start_datetime': 'date_time'}
)

overall_condition_selected = overall_condition_df[['person_id', 'condition_start_datetime']].rename(
    columns={'condition_start_datetime': 'date_time'}
)



# Combine all four
combined_overall_df = pd.concat([
    overall_condition_selected,
    overall_procedure_selected,
    symp_proc_selected,
    symp_diag_selected
], ignore_index=True)

combined_overall_df['date_time'] = pd.to_datetime(combined_overall_df['date_time'])

final_date_df = combined_overall_df.groupby('person_id', as_index=False).agg(
    final_date=('date_time', 'max')
)

# Display the result
final_date_df.info()

# Merge kidney stone datasets and final date datasets and get recurrence data

## Sort Colic dates

In [ ]:
%%R -i symptomatic_occurrence_diagnostic_codes

symptomatic_occurrence_diagnostic_codes <- as_tibble(symptomatic_occurrence_diagnostic_codes)
str(symptomatic_occurrence_diagnostic_codes)

In [ ]:
%%R

table(symptomatic_occurrence_diagnostic_codes$standard_concept_name)

In [ ]:
%%R

symptomatic_occurrence_diagnostic_codes <- symptomatic_occurrence_diagnostic_codes %>% mutate(
            initial_presentation = ifelse(
                standard_concept_name == "Calculus in pelviureteric junction" |
                standard_concept_name == "Calculus of kidney and ureter" |
                standard_concept_name == "Calculus of upper urinary tract" |
                standard_concept_name == "Hydronephrosis due to calculus of kidney and ureter" |
                standard_concept_name == "Renal colic" |
                standard_concept_name == "Ureteric colic" |
                standard_concept_name == "Ureteric stone of lower third of ureter" |
                standard_concept_name == "Ureteric stone" |
                standard_concept_name == "Urinary tract obstruction" |
                standard_concept_name == "Vesicoureteric junction calculus",
                "Colic",
                "Kidney_stone"
            )
)


#Calcium oxalate calculus of kidney 
#                 Calculus in pelviureteric junction 
#                      Calculus of kidney and ureter 
#                    Calculus of upper urinary tract 
#Hydronephrosis due to calculus of kidney and ureter 
#                                       Kidney stone 
#                                        Renal colic 
#                                     Ureteric colic 
#                                     Ureteric stone 
#            Ureteric stone of lower third of ureter 
#                           Uric acid renal calculus 
#                          Urinary tract obstruction 
#                                       Urolithiasis 
#                   Vesicoureteric junction calculus 

In [ ]:
%%R
table(symptomatic_occurrence_diagnostic_codes$initial_presentation)

## Sort Interventions

In [ ]:
%%R -i symptomatic_occurrence_procedure_codes

symptomatic_occurrence_procedure_codes <- as_tibble(symptomatic_occurrence_procedure_codes)
str(symptomatic_occurrence_procedure_codes)

In [ ]:
%%R 

table(symptomatic_occurrence_procedure_codes$standard_concept_name)

In [ ]:
%%R

symptomatic_occurrence_procedure_codes <- symptomatic_occurrence_procedure_codes %>% mutate(
            initial_presentation = ifelse(standard_concept_name == "Cystourethroscopy, with ureteroscopy and/or pyeloscopy (includes dilation of the ureter and/or pyeloureteral junction by any method); with lithotripsy (ureteral catheterization is included) (Deprecated)",
                                          "URS",
            ifelse(standard_concept_name == "Cystourethroscopy, with ureteroscopy and/or pyeloscopy (includes dilation of the ureter and/or pyeloureteral junction by any method); with removal or manipulation of calculus (ureteral catheterization is included) (Deprecated)",
                   "URS",
            ifelse(standard_concept_name == "Cystourethroscopy, with ureteroscopy and/or pyeloscopy; with lithotripsy including insertion of indwelling ureteral stent (eg, Gibbons or double-J type)",
                   "URS",
            ifelse(standard_concept_name == "Cystourethroscopy, with ureteroscopy and/or pyeloscopy; with removal or manipulation of calculus (ureteral catheterization is included)",
                   "URS",
            ifelse(standard_concept_name == "Extracorporeal shockwave lithotripsy",
                  "ESWL",
            ifelse(standard_concept_name == "Extracorporeal shockwave lithotripsy [ESWL] of the kidney, ureter and/or bladder",
                   "ESWL",
            ifelse(standard_concept_name == "Extracorporeal shockwave lithotripsy of calculus of kidney",
                   "ESWL",
            ifelse(standard_concept_name == "Extracorporeal shockwave lithotripsy of other sites",
                   "ESWL",
            ifelse(standard_concept_name == "Fragmentation in Left Ureter, Open Approach",
                  "Open",
            ifelse(standard_concept_name == "Fragmentation in Left Ureter, Via Natural or Artificial Opening Endoscopic",
                   "URS",
            ifelse(standard_concept_name == "Fragmentation in Right Ureter, External Approach",
                   "Open",
            ifelse(standard_concept_name == "Fragmentation in Right Ureter, Via Natural or Artificial Opening Endoscopic",
                   "URS",
            ifelse(standard_concept_name == "Nephrolithotomy for removal of calculus",
                   "PCNL",
            ifelse(standard_concept_name == "Nephrolithotomy for removal of staghorn calculus ",
                  "PCNL",
            ifelse(standard_concept_name == "Percutaneous extraction of kidney stone with fragmentation procedure",
                   "PCNL",
            ifelse(standard_concept_name == "Percutaneous nephrolithotomy or pyelolithotomy, lithotripsy, stone extraction, antegrade ureteroscopy, antegrade stent placement and nephrostomy tube placement, when performed, including imaging guidance; complex (eg, stone[s] > 2 cm, branching stones, st",
                   "PCNL",
            ifelse(standard_concept_name == "Percutaneous nephrolithotomy or pyelolithotomy, lithotripsy, stone extraction, antegrade ureteroscopy, antegrade stent placement and nephrostomy tube placement, when performed, including imaging guidance; simple (eg, stone[s] up to 2 cm in single location",
                   "PCNL",
            ifelse(standard_concept_name == "Pyelolithotomy",
                   "Open",
            ifelse(standard_concept_name == "Removal of calculus of renal pelvis through percutaneous nephrostomy",
                   "PCNL",
            ifelse(standard_concept_name == "Renal endoscopy through established nephrostomy or pyelostomy, with or without irrigation, instillation, or ureteropyelography, exclusive of radiologic service; with removal of foreign body or calculus",
                   "PCNL",
            ifelse(standard_concept_name == "Renal endoscopy through nephrotomy or pyelotomy, with or without irrigation, instillation, or ureteropyelography, exclusive of radiologic service; with removal of foreign body or calculus",
                   "PCNL",
            ifelse(standard_concept_name == "Renal endoscopy through nephrotomy or pyelotomy, with or without irrigation, instillation, or ureteropyelography, exclusive of radiologic service; with removal of foreign body or calculus",
                   "PCNL",
            ifelse(standard_concept_name == "Ureteral endoscopy through established ureterostomy, with or without irrigation, instillation, or ureteropyelography, exclusive of radiologic service; with removal of foreign body or calculus",
                    "URS",
            ifelse(standard_concept_name == "Ureterolithotomy, upper one-third of ureter",
                   "Open",
            ifelse(standard_concept_name == "Ureteroscopic laser fragmentation of ureteric calculus",
                   "URS",
            ifelse(standard_concept_name == "Ureteroscopy",
                   "URS",
            ifelse(standard_concept_name == "Ureteroscopy with biopsy ",
                   "URS",
                   NA
            
            )
            )
            )       
            )       
            )       
            )       
            )       
            )       
            )       
            )       
            )       
            )
            )
            )       
            )       
            )       
            )       
            )       
            )       
            )       
            )      
            )      
            )      
            )       
            )       
            )                              
            )
)

In [ ]:
%%R

symptomatic_occurrence_procedure_codes_only <- symptomatic_occurrence_procedure_codes %>% group_by(person_id)
str(symptomatic_occurrence_procedure_codes_only)

In [ ]:
symptomatic_occurrence_diagnostic_codes = %Rget symptomatic_occurrence_diagnostic_codes
symptomatic_occurrence_procedure_codes = %Rget symptomatic_occurrence_procedure_codes

In [ ]:
# Select and rename columns for symptomatic_occurrence_diagnostic_codes
diagnostic_selected = symptomatic_occurrence_diagnostic_codes[['person_id', 'condition_start_datetime', 'initial_presentation']]
diagnostic_selected = diagnostic_selected.dropna(subset=['initial_presentation'])
diagnostic_selected = diagnostic_selected.rename(columns={'condition_start_datetime': 'date_time'})

# Select and rename columns for symptomatic_occurrence_procedure_codes
procedure_selected = symptomatic_occurrence_procedure_codes[['person_id', 'procedure_datetime', 'initial_presentation']]
procedure_selected = procedure_selected.dropna(subset=['initial_presentation'])
procedure_selected = procedure_selected.rename(columns={'procedure_datetime': 'date_time'})

# Combine the DataFrames
combined_df = pd.concat([diagnostic_selected, procedure_selected], ignore_index=True)

# Display the result
combined_df.info()

In [ ]:
%%R -i combined_df

query_combined_df <- as_tibble(combined_df)
str(query_combined_df)

In [ ]:
%%R
intervention_pattern <- "PCNL|URS|ESWL|Open"

initial_intervention_df <- query_combined_df %>%
  
  # 1️⃣ Remove Kidney_stone codes
  filter(!grepl("Kidney_stone", initial_presentation)) %>%
  
  arrange(person_id, date_time) %>%
  group_by(person_id) %>%
  
  group_modify(~{
    
    df <- .x
    
    # First event after removing Kidney_stone
    first_event <- df[1, ]
    
    # ---- CASE A: First event is already an intervention ----
    if (grepl(intervention_pattern, first_event$initial_presentation)) {
      
      return(tibble(
        person_id = first_event$person_id,
        initial_intervention = first_event$initial_presentation,
        first_date = first_event$date_time
      ))
    }
    
    # ---- CASE B: First event is Colic ----
    if (grepl("Colic", first_event$initial_presentation)) {
      
      intervention_within_6m <- df %>%
        filter(
          date_time > first_event$date_time,
          date_time <= first_event$date_time %m+% months(6),
          grepl(intervention_pattern, initial_presentation)
        ) %>%
        slice_min(date_time, n = 1)
      
      # If intervention found within 6 months
      if (nrow(intervention_within_6m) > 0) {
        return(tibble(
          person_id = first_event$person_id,
          initial_intervention = intervention_within_6m$initial_presentation,
          first_date = intervention_within_6m$date_time
        ))
      }
      
      # If NO intervention within 6 months
      return(tibble(
        person_id = first_event$person_id,
        initial_intervention = first_event$initial_presentation,
        first_date = first_event$date_time
      ))
    }
    
    # Fallback (if some other code appears unexpectedly)
    return(tibble(
      person_id = first_event$person_id,
      initial_intervention = first_event$initial_presentation,
      first_date = first_event$date_time
    ))
    
  }) %>%
  
  ungroup()

table(initial_intervention_df$initial_intervention)

In [ ]:
%%R

str(initial_intervention_df)

In [ ]:
%%R

second_event_df <- initial_intervention_df %>%
  
  left_join(
    query_combined_df %>%
      filter(!grepl("Kidney_stone", initial_presentation)),
    by = "person_id"
  ) %>%
  
  filter(date_time >= first_date %m+% months(6)) %>%
  
  group_by(person_id) %>%
  summarise(
    second_date = min(date_time),
    second_event_flag = TRUE,
    .groups = "drop"
  )

final_df <- initial_intervention_df %>%
  left_join(second_event_df, by = "person_id") %>%
  mutate(
    second_event_flag = ifelse(is.na(second_event_flag), FALSE, second_event_flag)
  )

str(final_df)

In [ ]:
%%R

table(final_df$second_event_flag)

In [ ]:
%%R

# Remove Kidney_stone for consistency
clean_df <- query_combined_df %>%
  filter(!grepl("Kidney_stone", initial_presentation))

# Last follow-up per person
last_followup <- clean_df %>%
  group_by(person_id) %>%
  summarise(last_followup_date = max(date_time), .groups = "drop")

# Merge everything
km_df <- initial_intervention_df %>%
  left_join(second_event_df, by = "person_id") %>%
  left_join(last_followup, by = "person_id") %>%
  mutate(
    first_date = as.Date(first_date),
    second_date = as.Date(second_date),
    last_followup_date = as.Date(last_followup_date),
    
    event = ifelse(!is.na(second_date), 1, 0),
    
    end_date = ifelse(event == 1, second_date, last_followup_date),
    end_date = as.Date(end_date),
    
    time_days = as.numeric(end_date - first_date)
  )# %>%
  #filter(time_days > 0)

surv_object <- Surv(time = km_df$time_days, event = km_df$event)

km_fit <- survfit(
  Surv(time_days/365.25, event) ~ 1,
  data = km_df
)

ggsurvplot(
  km_fit,
  data = km_df,
  risk.table = TRUE,
  surv.median.line = "hv",
  conf.int = TRUE,
  break.time.by = 1,
  xlab = "Time (years)",
  ylab = "Survival probability",
  ggtheme = theme_bw(),
  xlim = c(0, 10),
  title = "Time to First Recurrence"
)


## Combine Colic + Interventions dataset

In [ ]:
%%R

symptomatic_occurrence_procedure_codes1 <- symptomatic_occurrence_procedure_codes %>% subset(select = c(person_id,
                                                                                                         procedure_datetime,
                                                                                                         initial_presentation)) %>% mutate(
date_time = procedure_datetime, .keep = "unused") 

symptomatic_occurrence_diagnostic_codes1 <- symptomatic_occurrence_diagnostic_codes %>% subset(select = c(person_id,
                                                                                                         condition_start_datetime,
                                                                                                         initial_presentation)) %>% mutate(
date_time = condition_start_datetime, .keep = "unused") 

combined_data <- rbind(symptomatic_occurrence_procedure_codes1,
                       symptomatic_occurrence_diagnostic_codes1) %>% as_tibble()

combined_data <- combined_data %>% arrange(desc(date_time))
 
str(combined_data)

## Get Symptomatic occurences dataset

In [ ]:
%%R

symptomatic_occurrence_procedure_codes2 <- symptomatic_occurrence_procedure_codes %>% 
    subset(select = c(person_id, 
                      procedure_datetime,
                      initial_presentation)) %>% 
    mutate(
        date_time = procedure_datetime, 
        .keep = "unused") 

symptomatic_occurrence_diagnostic_codes2 <- symptomatic_occurrence_diagnostic_codes %>% 
    subset(initial_presentation != "Kidney_stone") %>% 
    subset(select = c(person_id,
                      condition_start_datetime,
                      initial_presentation)) %>% 
    mutate(
        date_time = condition_start_datetime, 
        .keep = "unused")

combined_data2 <- rbind(symptomatic_occurrence_procedure_codes2,
                       symptomatic_occurrence_diagnostic_codes2) %>% as_tibble()

combined_data2$date_time <- anydate(combined_data2$date_time) 

combined_data2 <- combined_data2 %>% arrange(desc(date_time))
 
str(combined_data2)

In [ ]:
%%R

n_distinct(combined_data2$person_id)

In [ ]:
%%R -i overall_person_df_ethnicity

age_at_first_stone_df_1 <- as_tibble(overall_person_df_ethnicity) %>%
                            dplyr::select(person_id, date_of_birth)

str(age_at_first_stone_df_1)

In [ ]:
%%R

n_distinct(combined_data$person_id)

In [ ]:
%%R
n_distinct(combined_data2$person_id)

In [ ]:
%%R

symptomatic_episodes_data <- combined_data2 %>%
  distinct(person_id, date_time, initial_presentation) %>%
  mutate(initial_presentation = case_when(
      is.na(initial_presentation) ~ "Colic",
      TRUE ~ initial_presentation
  )) %>%
  group_by(person_id) %>%
  summarise(
    symptomatic_episode_date = list(sort(date_time)),
    initial_presentation = list(initial_presentation[order(date_time)])
  )

str(symptomatic_episodes_data)

In [ ]:
%%R

first_date_data <- symptomatic_episodes_data %>%
  mutate(
    first_date = map2(symptomatic_episode_date, initial_presentation, ~{
      dates <- .x
      procs <- .y
      
      if (is.na(procs[1]) || procs[1] != "Colic") {
        return(tibble(first_date = dates[1], initial_presentation = procs[1]))
      }
      
      within_6m <- which(
        !is.na(procs) &
        procs != "Colic" & 
        dates > dates[1] & 
        dates <= dates[1] %m+% months(6)
      )
      
      if (length(within_6m) > 0) {
        idx <- within_6m[1]
        return(tibble(first_date = dates[idx], initial_presentation = procs[idx]))
      }
      
      tibble(first_date = dates[1], initial_presentation = "Colic")
    })
  ) %>%
  select(-symptomatic_episode_date, -initial_presentation) %>%
  unnest(first_date)

str(first_date_data)

In [ ]:
%%R

table(first_date_data$initial_presentation)

In [ ]:
%%R

table(is.na(combined_data$initial_presentation))

In [ ]:
%%R

n_distinct(first_date_data$person_id)

## Remove Participants with only Kidney Stones coded

In [ ]:
%%R

combined_data_no_ks <- combined_data2 %>% 
    filter(initial_presentation != "Kidney_stone")
str(combined_data_no_ks)

## Get First dates dataset - first date of any stone code

In [ ]:
%%R

table(combined_data_no_ks$initial_presentation)

In [ ]:
%%R

table(combined_data$initial_presentation)

In [ ]:
%%R

str(combined_data2)

In [ ]:
%%R

first_dates <- first_date_data

str(first_dates)

In [ ]:
%%R
first_dates1 <- first_dates %>%
    left_join(age_at_first_stone_df_1, by = "person_id") 

first_dates1 <- first_dates1 %>%
    mutate(age_at_first_stone = (difftime(first_date, date_of_birth, units = "days")/365),
          .keep = "all")

first_dates1$age_at_first_stone <- as.numeric(first_dates1$age_at_first_stone)

str(first_dates1)

In [ ]:
%%R

hist(first_dates1$age_at_first_stone) 

## Get Overall dataset by joining those without kidney stones to first dates and symptomatic episode dates

In [ ]:
%%R

n_distinct(symptomatic_episodes_data$person_id)

In [ ]:
%%R

str(symptomatic_episodes_data)

In [ ]:
%%R

overall_data <- combined_data_no_ks %>%
  pivot_wider(names_from = initial_presentation, 
              values_from = date_time, 
              values_fn = list(date_time = ~list(.)))

str(overall_data)

In [ ]:
%%R

overall_data <- overall_data %>% subset(select = c(person_id,
                                                  Open,
                                                  PCNL,
                                                  Colic,
                                                  ESWL,
                                                  URS))

overall_data <- overall_data %>% left_join(first_dates1, by = "person_id")
overall_data <- symptomatic_episodes_data %>% left_join(overall_data, by=c("person_id" = "person_id"))

str(overall_data)

In [ ]:
%%R

nrow(overall_data)

In [ ]:
%%R -i overall_person_df_ethnicity

overall_person_df_ethnicity <- as_tibble(overall_person_df_ethnicity)

In [ ]:
%%R
overall_data %>% 
left_join(overall_person_df_ethnicity %>% select(person_id, sex_at_birth),
         by = "person_id") %>%
select(sex_at_birth) %>% table()

In [ ]:
%%R

str(overall_data)

In [ ]:
%%R

n_distinct(overall_data$person_id)

In [ ]:
%%R

overall_data_unique <- overall_data %>%
  group_by(person_id) %>%
  summarise(
    # initial_presentation: prefer non-Colic, fallback to Colic
    initial_presentation = if (any(initial_presentation != "Colic", na.rm = TRUE)) {
  first(na.omit(initial_presentation[initial_presentation != "Colic"]))
} else if (any(initial_presentation == "Colic", na.rm = TRUE)) {
  "Colic"
} else {
  "Colic"
},
    
    # Combine and keep all dates in list-columns
    symptomatic_episode_date = list(unique(unlist(symptomatic_episode_date))),
    Open = list(unique(unlist(Open))),
    PCNL = list(unique(unlist(PCNL))),
    Colic = list(unique(unlist(Colic))),
    ESWL = list(unique(unlist(ESWL))),
    URS = list(unique(unlist(URS))),
    
    # Keep first of duplicates
    date_of_birth = first(date_of_birth),
    first_date = min(first_date),
    age_at_first_stone = first(age_at_first_stone),
    
    .groups = "drop"
  ) %>%
  # Convert unlist'd numeric dates back to Date class
  mutate(
    across(c(symptomatic_episode_date, Open, PCNL, Colic, ESWL, URS),
           ~ map(.x, ~ if (length(.x) == 0 || all(is.na(.x))) NULL else as.Date(.x, origin = "1970-01-01")))
  )

In [ ]:
%%R

overall_data <- overall_data_unique

### Convert dates to POSIXct if not already done so

In [ ]:
%%R
overall_data$Open <- lapply(overall_data$Open, function(date_list) {
  converted_dates <- sapply(date_list, function(date) {
    if (!is.null(date) && !is.na(date) && is.character(date)) {
      # Convert to POSIXct using 'anytime'
      anytime(date)
    } else {
      NA
    }
  })
  
  if (length(converted_dates) == 0) {
    return(NA)
  } else {
    return(converted_dates)
  }
})

overall_data$Open <- lapply(overall_data$Open, function(x) {
  if (length(x) == 0) {
    return(NA)
  } else {
    return(ifelse(length(x) > 0, x[1], NA))
  }
})

overall_data$Open <- unlist(overall_data$Open, use.names = FALSE)
overall_data$Open <- as.POSIXct(overall_data$Open, origin = "1970-01-01", tz = "UTC")

head(overall_data)

In [ ]:
%%R
overall_data$PCNL <- lapply(overall_data$PCNL, function(date_list) {
  converted_dates <- sapply(date_list, function(date) {
    if (!is.null(date) && !is.na(date) && is.character(date)) {
      # Convert to POSIXct using 'anytime'
      anytime(date)
    } else {
      NA
    }
  })
  
  if (length(converted_dates) == 0) {
    return(NA)
  } else {
    return(converted_dates)
  }
})

overall_data$PCNL <- lapply(overall_data$PCNL, function(x) {
  if (length(x) == 0) {
    return(NA)
  } else {
    return(ifelse(length(x) > 0, x[1], NA))
  }
})

overall_data$PCNL <- unlist(overall_data$PCNL, use.names = FALSE)
overall_data$PCNL <- as.POSIXct(overall_data$PCNL, origin = "1970-01-01", tz = "UTC")

head(overall_data)

In [ ]:
%%R
overall_data$Colic <- lapply(overall_data$Colic, function(date_list) {
  converted_dates <- sapply(date_list, function(date) {
    if (!is.null(date) && !is.na(date) && is.character(date)) {
      # Convert to POSIXct using 'anytime'
      anytime(date)
    } else {
      NA
    }
  })
  
  if (length(converted_dates) == 0) {
    return(NA)
  } else {
    return(converted_dates)
  }
})

overall_data$Colic <- lapply(overall_data$Colic, function(x) {
  if (length(x) == 0) {
    return(NA)
  } else {
    return(ifelse(length(x) > 0, x[1], NA))
  }
})

overall_data$Colic <- unlist(overall_data$Colic, use.names = FALSE)
overall_data$Colic <- as.POSIXct(overall_data$Colic, origin = "1970-01-01", tz = "UTC")

head(overall_data)

In [ ]:
%%R
overall_data$ESWL <- lapply(overall_data$ESWL, function(date_list) {
  converted_dates <- sapply(date_list, function(date) {
    if (!is.null(date) && !is.na(date) && is.character(date)) {
      # Convert to POSIXct using 'anytime'
      anytime(date)
    } else {
      NA
    }
  })
  
  if (length(converted_dates) == 0) {
    return(NA)
  } else {
    return(converted_dates)
  }
})

overall_data$ESWL <- lapply(overall_data$ESWL, function(x) {
  if (length(x) == 0) {
    return(NA)
  } else {
    return(ifelse(length(x) > 0, x[1], NA))
  }
})

overall_data$ESWL <- unlist(overall_data$ESWL, use.names = FALSE)
overall_data$ESWL <- as.POSIXct(overall_data$ESWL, origin = "1970-01-01", tz = "UTC")

head(overall_data)

In [ ]:
%%R
overall_data$URS <- lapply(overall_data$URS, function(date_list) {
  converted_dates <- sapply(date_list, function(date) {
    if (!is.null(date) && !is.na(date) && is.character(date)) {
      # Convert to POSIXct using 'anytime'
      anytime(date)
    } else {
      NA
    }
  })
  
  if (length(converted_dates) == 0) {
    return(NA)
  } else {
    return(converted_dates)
  }
})


overall_data$URS <- lapply(overall_data$ESWL, function(x) {
  if (length(x) == 0) {
    return(NA)
  } else {
    return(ifelse(length(x) > 0, x[1], NA))
  }
})

overall_data$URS <- unlist(overall_data$URS, use.names = FALSE)
overall_data$URS <- as.POSIXct(overall_data$URS, origin = "1970-01-01", tz = "UTC")

head(overall_data)

### Find Initial_presentation

In [ ]:
%%R

# Function to count rows with non-empty lists (length > 1)
count_rows_with_data <- function(column) {
  sum(sapply(column, function(x) length(x) > 0))  # Count rows where list length > 1
}

# Apply the function to each column and get the count
data_count <- overall_data %>%
  summarise(across(everything(), count_rows_with_data))

# Print the count of rows with data for each column
print(data_count)

In [ ]:
%%R

procedure_dates <- symptomatic_occurrence_procedure_codes1 %>% drop_na(initial_presentation) %>% mutate(initial_procedure_date = "initial_procedure_date") %>% pivot_wider(
names_from = initial_procedure_date, 
              values_from = date_time, 
              values_fn = list(date_time = ~list(.))
)

procedure_dates$initial_procedure_date <- sapply(procedure_dates$initial_procedure_date, function(x) min(unlist(x), na.rm = TRUE))

procedure_dates$initial_procedure_date <- anydate(procedure_dates$initial_procedure_date)                                                 
                                                 
str(procedure_dates)

In [ ]:
%%R

nrow(overall_data)
table(procedure_dates$initial_presentation)

In [ ]:
%%R

procedure_dates_first <- procedure_dates %>%
  group_by(person_id) %>%
  slice_min(initial_procedure_date, n = 1, with_ties = FALSE) %>%
  ungroup()

overall_data2 <- overall_data %>%
  left_join(procedure_dates_first, by = "person_id") %>%
  mutate(
    initial_presentation = if_else(
      initial_presentation.x == "Colic" & !is.na(initial_presentation.y),
      initial_presentation.y,
      initial_presentation.x
    )
  ) %>%
  select(-initial_presentation.x, -initial_presentation.y) %>%
  distinct(person_id, .keep_all = TRUE) %>%
  mutate(
    initial_procedure_date = case_when(
        is.na(initial_procedure_date) ~ first(symptomatic_episode_date),
        TRUE ~ initial_procedure_date
    )
  ) %>%
  select(person_id, initial_presentation, initial_procedure_date, everything())

str(overall_data2)

In [ ]:
%%R

overall_data3 <- overall_data2 %>%
  rowwise() %>%
  mutate(
    initial_presentation = ifelse(!is.na(initial_presentation), 
                                 initial_presentation,
                                 "Colic"),
    initial_presentation_date = ifelse(!is.na(initial_procedure_date),
                                      initial_procedure_date,
                                      Colic)
  ) %>%
  ungroup() %>% subset(select = c(person_id,
                                 initial_presentation,
                                 initial_procedure_date,
                                 symptomatic_episode_date))



# View the updated dataset
str(overall_data3)

In [ ]:
%%R

table(overall_data3$initial_presentation)

## Create KM plot

In [ ]:
%%R

ggplot(first_dates1, aes(x = first_date)) +
  geom_histogram(binwidth = 365 * 10, fill = "skyblue", color = "black", alpha = 0.7) + # binwidth = 3 years
  scale_x_datetime(date_labels = "%Y-%m-%d", date_breaks = "1 year") +
  labs(
    title = "Histogram of First Dates",
    x = "First Date",
    y = "Frequency"
  ) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

In [ ]:
%%R

str(first_dates1)

In [ ]:
%%R

overall_data4 <- first_dates1 %>% 
    left_join(overall_data3 %>% dplyr::select(-initial_presentation), by = "person_id") 

overall_data4$first_date <- anydate(overall_data4$first_date)

overall_data4 <- overall_data4 %>% mutate(
    initial_presentation = ifelse(!is.na(initial_presentation),
                                 initial_presentation,
                                 "Kidney_stone"),
    first_date = anydate(ifelse(!is.na(initial_procedure_date),
                       anydate(initial_procedure_date),
                       anydate(first_date)))
)

overall_data4 <- overall_data4 %>% mutate(
symptomatic_episode_date = lapply(ifelse(symptomatic_episode_date == "NULL",
                                     anydate(first_date),
                                     symptomatic_episode_date), anydate)
)

overall_data4$initial_presentation <- as.factor(overall_data4$initial_presentation)

str(overall_data4)

In [ ]:
%%R

table(overall_data4$initial_presentation)

In [ ]:
%%R

overall_data5 <- overall_data4 %>%
  mutate(
    symptomatic_episode_date = map2(symptomatic_episode_date, first_date, ~ .x[.x >= .y])
  )

str(overall_data5)

In [ ]:
%%R

table(overall_data5$initial_presentation)

In [ ]:
%%R -i final_date_df

final_dates <- as_tibble(final_date_df)
final_dates$final_date <- anydate(final_dates$final_date)
str(final_dates)

In [ ]:
%%R

hist(final_dates$final_date, breaks = 10)

In [ ]:
%%R

str(overall_data4)

table(is.na(overall_data4$initial_procedure_date), overall_data4$initial_presentation)

In [ ]:
%%R

overall_data4 %>%
  mutate(dates_match = map2_lgl(symptomatic_episode_date, first_date, ~ .y %in% .x)) %>%
  count(dates_match)

In [ ]:
%%R
str(overall_data4)

In [ ]:
%%R

km_recurrence <- overall_data4 %>%
  left_join(final_dates, by = "person_id") %>%
  mutate(
    # Count the number of recurrences
    recurrence = map_int(symptomatic_episode_date, ~ {
      sorted_dates <- sort(.x)
      if(length(sorted_dates) < 2 || all(is.na(sorted_dates))) {
        return(0)  # No recurrences if there are fewer than 2 dates
      }
      
      date_diffs <- diff(sorted_dates)
      sum(date_diffs >= 180)  # 180 days is about 6 months
    }),
    
    # Calculate the time-to-event from the first to second date >= 6 months apart
    time_to_event = map2_dbl(symptomatic_episode_date, final_date, ~ {
      sorted_dates <- sort(.x)
      
      # Handle case where there's only one date in symptomatic_episode_date
      if(length(sorted_dates) == 1) {
        if(identical(.y, sorted_dates[1])) {
          # If final_date equals the symptomatic_episode_date, use today's date for time at risk
          time_at_risk <- as.numeric(Sys.Date() - sorted_dates[1])
        } else {
          # If there's only one date, use the difference between final_date and symptomatic_episode_date
          time_at_risk <- as.numeric(.y - sorted_dates[1])
        }
        return(time_at_risk)  # Return time-at-risk in days
      }
      
      # Check if there are at least 2 dates for comparing events
      if(length(sorted_dates) >= 2 && !all(is.na(sorted_dates))) {
        # Find the first pair of dates >= 6 months apart
        date_diffs <- diff(sorted_dates)
        idx <- which(date_diffs >= 180)  # 180 days is about 6 months
        
        if(length(idx) > 0) {
          # Calculate the time to event (in days) between the first and second date
          return(as.numeric(sorted_dates[idx[1] + 1] - sorted_dates[idx[1]]))
        }
      }
      
      # If no pair of dates >= 6 months apart, calculate time at risk using final_date
      first_date <- sorted_dates[1]
      time_at_risk <- as.numeric(.y - first_date)
      return(time_at_risk)  # Time-at-risk (days) from the first event date
    })
  )


km_recurrence <- km_recurrence %>% subset(initial_presentation != "Kidney_stone")
km_recurrence$pheno <- ifelse(km_recurrence$recurrence > 0, 1, 0)
km_recurrence$pheno <- as.numeric(km_recurrence$pheno)
km_recurrence$time_to_event_yr <- (km_recurrence$time_to_event / 365)

# Check the structure of the updated data
str(km_recurrence)

In [ ]:
%%R

table(km_recurrence$recurrence)

In [ ]:
%%R

plot_missing(km_recurrence)

In [ ]:
%%R

str(km_recurrence)

In [ ]:
%%R

km_recurrence %>%
  mutate(fu_time = as.numeric(final_date - first_date) / 365.25) %>%
  summarise(
    mean_fu_time = mean(fu_time),
    sd_fu_time = sd(fu_time),
    median_fu_time = median(fu_time),
    IQR_q25 = quantile(fu_time, 0.25),
    IQR_q75 = quantile(fu_time, 0.75),
    min = quantile(fu_time, 0),
    max = quantile(fu_time, 1)
  )

In [ ]:
%%R

hist((km_recurrence %>%
  mutate(fu_time = as.numeric(final_date - first_date) / 365.25))$fu_time
    )

In [ ]:
%%R

km_recurrence1 <- km_recurrence %>%
    dplyr::select(-symptomatic_episode_date) %>%
    group_by(person_id) %>%
    summarise(
         pheno = if_else(any(pheno == 1), 1, 0),
        initial_presentation,
        time_to_event = min(time_to_event),
        .groups = "drop"
    )

km_fit <- surv_fit(Surv(time_to_event/365.25, pheno) ~ 1, data=km_recurrence1) 
km_overall <- ggsurvplot(
  km_fit,
  data = km_recurrence1,
  risk.table = TRUE,
  surv.median.line = "hv",
  conf.int = TRUE,
  tables.theme = theme_cleantable(),
  break.time.by = 4,               # Change from 1 to 2 years (0, 2, 4, 6, 8, 10)
  xlab = "Time (years)",
  xlim = c(0, 25),
  title = "Time to First Recurrence",
  pval = TRUE,
  censor = FALSE,
  legend.title = "Initial Presentation",
  legend = "right",                # Move legend to right to avoid overlap
  font.main = 14,
  font.x = 12,
  font.y = 12,
  font.tickslab = 10,
  fontsize = 3.5,                  # Risk table font size
  ggtheme = theme_bw()
)

km_overall

In [ ]:
%%R

str(km_recurrence1)
n_distinct(km_recurrence1$person_id)
table(km_recurrence1$pheno)

In [ ]:
%%R

km_recurrence1 %>%
    ggplot(aes(x = time_to_event/365.25, fill = as.factor(pheno))) +
    geom_histogram(binwidth = 0.5) +
  labs(x = "Time to event (years)", fill = "Recurrence")

In [ ]:
%%R

hist(km_recurrence$first_date, breaks = 50)

In [ ]:
%%R

km_fit <- surv_fit(Surv(time_to_event/365.25, pheno) ~ initial_presentation, data=km_recurrence1) 
km_split_by_initial_operation <- ggsurvplot(
  km_fit,
  data = km_recurrence1,
  risk.table = TRUE,
  risk.table.height = 0.35,        # Taller risk table for 5 groups
  surv.median.line = "hv",
  conf.int = TRUE,
  tables.theme = theme_cleantable(),
  break.time.by = 4,               # Change from 1 to 2 years (0, 2, 4, 6, 8, 10)
  xlab = "Time (years)",
  xlim = c(0, 20),
  title = "Time to First Recurrence",
  pval = TRUE,
  censor = FALSE,
  legend.title = "Initial Presentation",
  legend = "right",                # Move legend to right to avoid overlap
  font.main = 14,
  font.x = 12,
  font.y = 12,
  font.tickslab = 10,
  fontsize = 3.5,                  # Risk table font size
  ggtheme = theme_minimal()
)

km_split_by_initial_operation

# Plot Event number vs Mean time between events

## Prepare Data

### Overall dataset

In [ ]:
%%R

str(km_recurrence)

In [ ]:
%%R

time_between_events <- km_recurrence %>%
  unnest(symptomatic_episode_date) %>%
  group_by(person_id) %>%
  arrange(symptomatic_episode_date) %>%
  mutate(
    date_diff = as.numeric(difftime(symptomatic_episode_date, lag(symptomatic_episode_date), units = "days"))
  ) %>%
  summarise(
    number_of_events = n(),
    number_of_valid_gaps = sum(date_diff > 183, na.rm = TRUE),
    mean_time_between_events = mean(date_diff[date_diff > 183], na.rm = TRUE),
    time_at_risk_days = as.numeric(max(symptomatic_episode_date) - min(symptomatic_episode_date)),
    mean_time_or_risk = ifelse(number_of_valid_gaps == 0, time_at_risk_days, mean_time_between_events),
    .groups = "drop"
  )

## Overall

In [ ]:
%%R

#### Plot 1 ####

overall_recurrence_rate_plot <- ggplot(
  time_between_events %>%
    filter(!is.na(mean_time_or_risk) &
             is.finite(mean_time_or_risk)),
  aes(
    x = number_of_valid_gaps,
    y = mean_time_or_risk / 30.44
  )
) +
  geom_point(alpha = 0.3, color = "steelblue", size = 2) +
  labs(
    title = "Overall Data",
    x = "Number of Recurrences (>6 months apart)",
    y = "Mean Time Between Events or Time-at-Risk (months)"
  ) +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5, size = 14, face = "bold"),
    axis.title = element_text(size = 12)
  ) + ylim(0,400)

overall_recurrence_rate_plot

In [ ]:
%%R

overall_recurrence_rate_plot_box <- ggplot(
  time_between_events %>%
    filter(!is.na(mean_time_or_risk) &
             is.finite(mean_time_or_risk)),
  aes(
    x = factor(number_of_valid_gaps),
    y = mean_time_or_risk / 30.44
  )
) +
  geom_boxplot(fill = "steelblue", alpha = 0.7, outlier.alpha = 0.3) +
  labs(
    title = "Overall Data",
    x = "Number of Recurrences (>6 months apart)",
    y = "Mean Time Between Events or Time-at-Risk (months)"
  ) +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5, size = 14, face = "bold"),
    axis.title = element_text(size = 12)
  ) + ylim(0, 400)

overall_recurrence_rate_plot_box

## Within 5 Years

In [ ]:
%%R

time_between_events_5_yr <- km_recurrence %>%
  unnest(symptomatic_episode_date) %>%
  group_by(person_id) %>%
  arrange(symptomatic_episode_date) %>%
  
  # Define 5-year window from first event
  mutate(
    first_event_date = min(symptomatic_episode_date, na.rm = TRUE),
    end_of_5yr_followup = first_event_date + years(5)
  ) %>%
  
  # Keep only events within 5 years of first event
  filter(symptomatic_episode_date <= end_of_5yr_followup) %>%
  
  # Calculate time differences
  mutate(
    date_diff = as.numeric(difftime(symptomatic_episode_date, lag(symptomatic_episode_date), units = "days"))
  ) %>%
  
  summarise(
    number_of_events_5yr = n(),
    number_of_valid_gaps_5yr = sum(date_diff > 183, na.rm = TRUE),
    mean_time_between_events_5yr = mean(date_diff[date_diff > 183], na.rm = TRUE),
    time_at_risk_days = as.numeric(first(end_of_5yr_followup) - first(first_event_date)),
    mean_time_or_risk_5yr = ifelse(
      number_of_valid_gaps_5yr == 0,
      time_at_risk_days,
      mean_time_between_events_5yr
    ),
    .groups = "drop"
  )

In [ ]:
%%R

basic_5_yr_rec_rates_plot <- ggplot(
  time_between_events_5_yr %>%
    filter(!is.na(mean_time_or_risk_5yr) &
             is.finite(mean_time_or_risk_5yr)),
  aes(
    x = number_of_valid_gaps_5yr,
    y = mean_time_or_risk_5yr / 30.44
  )
) +
  geom_point(alpha = 0.3, color = "steelblue", size = 2) +
  labs(
    title = "Within 5 Years",
    x = "Number of Recurrences (>6 months apart)",
    y = "Mean Time Between Events or Time-at-Risk (months)"
  ) +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5, size = 14, face = "bold"),
    axis.title = element_text(size = 12)
  ) +
  scale_x_continuous(
    limits = c(0, 5),
    breaks = seq(0, 6, by = 1)
  ) + ylim(0,62)

basic_5_yr_rec_rates_plot

In [ ]:
%%R

basic_5_yr_rec_rates_plot_box <- ggplot(
  time_between_events_5_yr %>%
    filter(!is.na(mean_time_or_risk_5yr) &
             is.finite(mean_time_or_risk_5yr)),
  aes(
    x = factor(number_of_valid_gaps_5yr),
    y = mean_time_or_risk_5yr / 30.44
  )
) +
  geom_boxplot(fill = "steelblue", alpha = 0.7, outlier.alpha = 0.3) +
  labs(
    title = "Within 5 Years",
    x = "Number of Recurrences (>6 months apart)",
    y = "Mean Time Between Events or Time-at-Risk (months)"
  ) +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5, size = 14, face = "bold"),
    axis.title = element_text(size = 12)
  ) +
  scale_x_discrete(limits = as.character(0:5)) +
  ylim(0, 62)

basic_5_yr_rec_rates_plot_box

## Within 10 Years

In [ ]:
%%R

time_between_events_10_yr <- km_recurrence %>%
  unnest(symptomatic_episode_date) %>%
  group_by(person_id) %>%
  arrange(symptomatic_episode_date) %>%
  
  # Define 10-year window from first event
  mutate(
    first_event_date = min(symptomatic_episode_date, na.rm = TRUE),
    end_of_10yr_followup = first_event_date + years(10)
  ) %>%
  
  # Keep only events within 10 years of first event
  filter(symptomatic_episode_date <= end_of_10yr_followup) %>%
  
  # Calculate time differences
  mutate(
    date_diff = as.numeric(difftime(symptomatic_episode_date, lag(symptomatic_episode_date), units = "days"))
  ) %>%
  
  summarise(
    number_of_events_10yr = n(),
    number_of_valid_gaps_10yr = sum(date_diff > 183, na.rm = TRUE),
    mean_time_between_events_10yr = mean(date_diff[date_diff > 183], na.rm = TRUE),
    time_at_risk_days = as.numeric(first(end_of_10yr_followup) - first(first_event_date)),
    mean_time_or_risk_10yr = ifelse(
      number_of_valid_gaps_10yr == 0,
      time_at_risk_days,
      mean_time_between_events_10yr
    ),
    .groups = "drop"
  )

In [ ]:
%%R

basic_10_yr_rec_rates_plot <- ggplot(
  time_between_events_10_yr %>%
    filter(!is.na(mean_time_or_risk_10yr) &
             is.finite(mean_time_or_risk_10yr)),
  aes(
    x = number_of_valid_gaps_10yr,
    y = mean_time_or_risk_10yr / 30.44
  )
) +
  geom_point(alpha = 0.3, color = "steelblue", size = 2) +
  labs(
    title = "Within 10 Years",
    x = "Number of Recurrences (>6 months apart)",
    y = "Mean Time Between Events or Time-at-Risk (months)"
  ) +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5, size = 14, face = "bold"),
    axis.title = element_text(size = 12)
  ) +
  scale_x_continuous(
    limits = c(0, 11),
    breaks = seq(0, 11, by = 1)
  ) + ylim(0,121)

basic_10_yr_rec_rates_plot

In [ ]:
%%R

basic_10_yr_rec_rates_plot_box <- ggplot(
  time_between_events_10_yr %>%
    filter(!is.na(mean_time_or_risk_10yr) &
             is.finite(mean_time_or_risk_10yr)),
  aes(
    x = factor(number_of_valid_gaps_10yr),
    y = mean_time_or_risk_10yr / 30.44
  )
) +
  geom_boxplot(fill = "steelblue", alpha = 0.7, outlier.alpha = 0.3) +
  labs(
    title = "Within 10 Years",
    x = "Number of Recurrences (>6 months apart)",
    y = "Mean Time Between Events or Time-at-Risk (months)"
  ) +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5, size = 14, face = "bold"),
    axis.title = element_text(size = 12)
  ) +
  scale_x_discrete(limits = as.character(0:11)) +
  ylim(0, 121)

basic_10_yr_rec_rates_plot_box

## Combined Basic Plot

In [ ]:
%%R

overall_recurrence_rate_plot +
  basic_5_yr_rec_rates_plot + 
  basic_10_yr_rec_rates_plot + 
  plot_annotation(tag_levels = "A",
                  title = "Mean Time Between Events vs Number of Recurrences") + 
  plot_layout(axes = "collect")


In [ ]:
%%R

recurrence_rate_boxes <- overall_recurrence_rate_plot_box +
  basic_5_yr_rec_rates_plot_box + 
  basic_10_yr_rec_rates_plot_box + 
  plot_annotation(
    tag_levels = list(c("D", "E", "F")),
    title = "All of Us"
  ) + 
  plot_layout(axes = "collect")

recurrence_rate_boxes

## Save Images

In [ ]:
import os
bucket = os.getenv("WORKSPACE_BUCKET")
bucket

In [ ]:
%%R

saveRDS(overall_recurrence_rate_plot, "overall_recurrence_rate_plot_aou.rds")
saveRDS(basic_5_yr_rec_rates_plot, "basic_5_yr_rec_rates_plot_aou.rds")
saveRDS(basic_10_yr_rec_rates_plot, "basic_10_yr_rec_rates_plot_aou.rds")
saveRDS(overall_recurrence_rate_plot_box, "overall_recurrence_rate_plot_aou_box.rds")
saveRDS(basic_5_yr_rec_rates_plot_box, "basic_5_yr_rec_rates_plot_aou_box.rds")
saveRDS(basic_10_yr_rec_rates_plot_box, "basic_10_yr_rec_rates_plot_aou_box.rds")

In [ ]:
!gsutil -m cp overall_recurrence_rate_plot_aou.rds {bucket}/data/eau_risk/
!gsutil -m cp basic_5_yr_rec_rates_plot_aou.rds {bucket}/data/eau_risk/
!gsutil -m cp basic_10_yr_rec_rates_plot_aou.rds {bucket}/data/eau_risk/
!gsutil -m cp overall_recurrence_rate_plot_aou_box.rds {bucket}/data/eau_risk/
!gsutil -m cp basic_5_yr_rec_rates_plot_aou_box.rds {bucket}/data/eau_risk/
!gsutil -m cp basic_10_yr_rec_rates_plot_aou_box.rds {bucket}/data/eau_risk/

In [ ]:
%%R

overall_recurrence_rate_plot 

# Explore EAU risk stratification

## Download data

In [ ]:
import os
bucket = os.getenv("WORKSPACE_BUCKET")
bucket

In [ ]:
!gsutil -m cp {bucket}/data/eau_risk.csv .

In [ ]:
%%R

eau_risk <- read.csv("eau_risk.csv") %>% as_tibble()
str(eau_risk)

In [ ]:
%%R

str(overall_data)

In [ ]:
%%R -i overall_person_df

overall_person_df1 <- as_tibble(overall_person_df)

age_at_first_stone_df <- combined_data %>%
    group_by(
      person_id
    ) %>%
    summarise(
        first_date = min(date_time)
    ) %>%
    ungroup() %>%
    left_join(overall_person_df, by = "person_id") %>%
    mutate(
        age_at_first_stone = as.numeric((first_date - date_of_birth)/365.25)
    )

str(age_at_first_stone_df)

In [ ]:
%%R

mean(age_at_first_stone_df$age_at_first_stone, na.rm = TRUE)

In [ ]:
%%R
sd(age_at_first_stone_df$age_at_first_stone, na.rm = TRUE)

In [ ]:
%%R
quantile(age_at_first_stone_df$age_at_first_stone, 0.25, na.rm = TRUE)

In [ ]:
%%R
quantile(age_at_first_stone_df$age_at_first_stone, 0.75, na.rm = TRUE)

In [ ]:
%%R

hist(age_at_first_stone_df$age_at_first_stone, na.rm=TRUE)

In [ ]:
%%R 

age_at_first_stone_df %>% 
    filter(person_id %in% km_recurrence$person_id) %>%
    summarise(
        n = n(),
        mean_age = mean(age_at_first_stone, na.rm = TRUE),
        age_sd = sd(age_at_first_stone, na.rm = TRUE)
        )

In [ ]:
%%R

n_distinct(eau_risk$person_id)

## Sort data + merge with km_recurrence

In [ ]:
%%R

eau_risk$date_time <- as.Date(eau_risk$date_time)

In [ ]:
%%R

eau_risk <- eau_risk %>%
    group_by(person_id) %>%
    dplyr::select(-standard_concept_name) %>%
    arrange(date_time)

str(eau_risk)

In [ ]:
%%R 

str(km_recurrence)

In [ ]:
%%R

mean(km_recurrence$age_at_first_stone, na.rm = TRUE)

In [ ]:
%%R
sd(km_recurrence$age_at_first_stone, na.rm=TRUE)

## Save plots

In [ ]:
!gsutil -m cp roc_plot_aou.rds {bucket}/data/eau_risk/
!gsutil -m cp first_to_second_event_by_risk_aou.rds {bucket}/data/eau_risk/
#!gsutil -m cp km_split_by_initial_operation_aou.rds {bucket}/data/eau_risk/

In [ ]:
!gsutil -m ls {bucket}/data/eau_risk/

In [ ]:
!gsutil -m cp -Z -r {bucket}/data/eau_risk/* {bucket}/data/eau_risk_gzipped/

In [ ]:
!gsutil -m cp -r {bucket}/data/eau_risk .

In [ ]:
!gzip -r eau_risk/

In [ ]:
!gsutil -m cp -r eau_risk/ {bucket}/data/eau_risk_gzipped/

In [ ]:
!gsutil -m ls {bucket}/data/eau_risk_gzipped/eau_risk/

# Table of EAU risk Categories

## Import data

In [ ]:
%%R

km_recurrence %>% filter(time_to_event_yr > 0.5) %>% select(initial_presentation) %>% table()

In [ ]:
%%R

str(eau_risk)

In [ ]:
%%R

table(eau_risk$standard_concept_name)

## Create Risk categories

In [ ]:
%%R

pmhx_cleaned <- eau_risk %>%
  filter(person_id %in% km_recurrence$person_id) %>%
  left_join(km_recurrence %>% select(person_id, first_date), 
            by = "person_id",
            relationship = "many-to-many") %>%
  filter(date_time < first_date) %>%
  group_by(person_id, first_date) %>%
  summarise(standard_concept_names = list(unique(standard_concept_name)), .groups = "drop") %>%
  mutate(
    dm = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Diabetes mellitus", "Diabetes mellitus without complication",
      "Type 1 diabetes mellitus", "Type 1 diabetes mellitus without complication",
      "Type 2 diabetes mellitus", "Type 2 diabetes mellitus without complication",
      "Type 2 diabetes mellitus with ulcer", "Type 1 diabetes mellitus with ulcer",
      "Drug-induced diabetes mellitus", "Steroid-induced diabetes",
      "Gestational diabetes mellitus", "Gestational diabetes mellitus class A-1",
      "Gestational diabetes mellitus class A-2",
      "Secondary diabetes mellitus", "Secondary endocrine diabetes mellitus",
      "Posttransplant diabetes mellitus", "Neonatal diabetes mellitus",
      "Latent autoimmune diabetes mellitus in adult",
      "Maturity-onset diabetes of the young",
      "Maturity onset diabetes of the young, type 1",
      "Maturity onset diabetes of the young, type 2",
      "Maturity-onset diabetes of the young, type 3",
      "Maturity-onset diabetes of the young, type 5",
      "Insulin treated type 2 diabetes mellitus",
      "Insulin dependent diabetes mellitus type 1A",
      "Insulin dependent diabetes mellitus type 1B",
      "Ketosis-prone diabetes mellitus", "Lipoatrophic diabetes",
      "Diabetes mellitus associated with cystic fibrosis",
      "Diabetes mellitus associated with pancreatic disease",
      "Diabetes mellitus due to cystic fibrosis",
      "Diabetes mellitus due to pancreatic injury",
      "Diabetes mellitus in mother complicating pregnancy, childbirth AND/OR puerperium",
      "Pre-existing type 1 diabetes mellitus in pregnancy",
      "Pre-existing type 2 diabetes mellitus in pregnancy",
      "Pre-existing diabetes mellitus in pregnancy",
      "Type II diabetes mellitus in remission"
    )))),

    hyperpth = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Hyperparathyroidism", "Primary hyperparathyroidism",
      "Secondary hyperparathyroidism", "Tertiary hyperparathyroidism",
      "Secondary hyperparathyroidism of nonrenal origin",
      "Hyperparathyroidism due to renal insufficiency",
      "Hyperparathyroidism due to vitamin D deficiency",
      "Hyperparathyroidism due to lithium therapy",
      "Familial hyperparathyroidism",
      "Parathyroidectomy", "Complete parathyroidectomy",
      "Subtotal parathyroidectomy", "Parathyroid adenoma excision",
      "Total parathyroidectomy and parathyroid tissue transplantation",
      "Partial parathyroidectomy and parathyroid tissue transposition"
    )))),

    uti = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Urinary tract infectious disease", "Acute urinary tract infection",
      "Chronic urinary tract infection", "Recurrent urinary tract infection",
      "Lower urinary tract infectious disease", "Upper urinary tract infection",
      "Acute upper urinary tract infection", "Acute lower urinary tract infection",
      "Febrile urinary tract infection", "Postoperative urinary tract infection",
      "Catheter-associated urinary tract infection",
      "Urinary tract infection in pregnancy",
      "Urinary tract infection caused by Escherichia coli",
      "Urinary tract infection caused by Enterococcus",
      "Urinary tract infection caused by Klebsiella",
      "Urinary tract infection caused by Pseudomonas",
      "Acute pyelonephritis", "Acute pyelonephritis without medullary necrosis",
      "Acute pyelonephritis with medullary necrosis",
      "Chronic pyelonephritis", "Chronic obstructive pyelonephritis",
      "Chronic pyelonephritis without medullary necrosis",
      "Chronic pyelonephritis with medullary necrosis",
      "Pyelonephritis", "Pyelonephritis in pregnancy",
      "Pyonephrosis", "Infective cystitis", "Infective urethritis",
      "Acute infective cystitis", "Recurrent infective cystitis",
      "Bacterial urinary infection", "Neonatal urinary tract infection",
      "Emphysematous pyelonephritis", "Emphysematous cystitis",
      "Infections of kidney in pregnancy", "Infections of bladder in pregnancy",
      "Infectious disorder of kidney", "Non-obstructive reflux-associated chronic pyelonephritis"
    )))),

    immunosuppression = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Immunosuppression"
    )))),

    transplant_immunosuppression = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Transplanted kidney present", "Transplanted heart present",
      "Transplanted liver present", "Transplanted lung present",
      "Transplanted heart-lung present", "Transplanted cornea present",
      "Transplant of kidney", "Transplantation of heart",
      "Transplantation of liver", "Transplantation of pancreas",
      "Transplantation of lung", "Transplant present",
      "Transplant nephrectomy", "Donor renal transplantation",
      "Live donor renal transplant", "Cadaveric renal transplant",
      "Homotransplant of pancreas", "Orthotopic liver transplant",
      "Heterotopic liver transplant", "Double lung transplant",
      "Single lung transplant", "Autotransplantation of kidney",
      "Renal transplant rejection", "Cardiac transplant rejection",
      "Lung transplant rejection", "Liver transplant rejection",
      "Bone marrow transplant rejection", "Bone marrow transplant present",
      "Bone marrow transplant failure", "Disorder of transplanted kidney",
      "Disorder of transplanted heart", "Disorder of transplanted bone marrow",
      "Complication of transplanted liver", "Complication of transplanted lung",
      "Complication of transplanted pancreas", "Complication of transplanted intestines",
      "Failed renal transplant", "Cardiac transplant failure",
      "Liver transplant failure", "Liver transplant disorder",
      "Heart-lung transplant failure and rejection",
      "Transplanted organ failure", "Transplanted organ rejection",
      "Chronic rejection of renal transplant", "Acute rejection of renal transplant",
      "Accelerated rejection of renal transplant",
      "Delayed renal graft function", "BK virus nephropathy",
      "Transplant glomerulopathy", "Transplant renal artery stenosis",
      "Recurrent post-transplant renal disease",
      "Arteriosclerosis of coronary artery bypass graft of transplanted heart",
      "Coronary arteriosclerosis in artery of transplanted heart",
      "Hypertension secondary to kidney transplant",
      "Hypertension associated with transplantation"
    )))),

    htn = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Essential hypertension", "Benign essential hypertension",
      "Benign hypertension", "Malignant essential hypertension",
      "Malignant hypertension", "Secondary hypertension",
      "Benign secondary hypertension", "Malignant secondary hypertension",
      "Renovascular hypertension", "Renal hypertension",
      "Benign secondary renovascular hypertension",
      "Malignant secondary renovascular hypertension",
      "Hypertensive disorder", "Hypertensive crisis",
      "Hypertensive emergency", "Hypertensive urgency",
      "Pregnancy-induced hypertension", "Maternal hypertension",
      "Hypertension secondary to endocrine disorder",
      "Hypertension complicating pregnancy, childbirth and the puerperium",
      "Pre-existing hypertension complicating pregnancy, childbirth and puerperium",
      "Labile essential hypertension", "Labile systemic arterial hypertension",
      "Resistant hypertensive disorder", "Paroxysmal hypertension",
      "Diastolic hypertension", "Systolic hypertension",
      "Systolic essential hypertension", "Postoperative hypertension",
      "Transient hypertension", "Intermittent hypertension"
    )))),

    dyslipidaemia = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Dyslipidemia", "Complex dyslipidemia",
      "Dyslipidemia due to type 1 diabetes mellitus",
      "Dyslipidemia due to type 2 diabetes mellitus",
      "Dyslipidemia with high density lipoprotein below reference range and triglyceride above reference range due to type 2 diabetes mellitus",
      "Metabolic syndrome X"
    )))),

    pkd = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Autosomal dominant polycystic kidney disease",
      "Autosomal dominant polycystic kidney disease in childhood",
      "Polycystic kidney disease, infantile type",
      "Acquired polycystic kidney disease",
      "Multiple congenital cysts of kidney", "Multiple renal cysts",
      "Medullary sponge kidney", "Bilateral medullary sponge kidney",
      "Multicystic renal dysplasia", "Infected renal cyst"
    )))),

    msk = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Lumbar sprain", "Cervical spine sprain", "Thoracic back sprain",
      "Sprain of spinal ligament", "Sprain of ligament of lumbosacral joint",
      "Sprain of sacrococcygeal ligament",
      "Annular tear of lumbar disc", "Intervertebral disc rupture",
      "Rupture of lumbar intervertebral disc",
      "Tear of the annulus fibrosus of intervertebral disc",
      "Fatigue fracture of vertebra", "Osteoporotic fracture of vertebra",
      "Osteoporotic vertebral collapse", "Pathological fracture of vertebra"
    )))),

    bowel_surgery = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Total colectomy", "Laparoscopic total colectomy",
      "Right colectomy", "Left colectomy", "Extended right hemicolectomy",
      "Sigmoid colectomy", "Ileocolic resection", "Rectosigmoidectomy",
      "Abdominoperineal resection", "Anterior resection of rectum with colostomy",
      "Low anterior resection of rectum", "Hartmann operation, rectal resection",
      "Partial colectomy with anastomosis", "Partial colectomy with coloproctostomy",
      "Partial colectomy with coloproctostomy and colostomy",
      "Partial resection of colon", "Partial excision of large intestine",
      "Large intestine excision", "Excision of colon", "Excision of colon and rectum",
      "Laparoscopic excision of part of colon",
      "Laparoscopic-assisted right colectomy", "Laparoscopic-assisted sigmoid colectomy",
      "Laparoscopic left hemicolectomy", "Hand assisted laparoscopic left colectomy",
      "Small intestine excision", "Total excision of small intestine",
      "Laparoscopic excision of small intestine",
      "Multiple segmental resections of small intestine",
      "Resection of exteriorized segment of small intestine",
      "Pancreaticoduodenectomy", "Radical pancreaticoduodenectomy",
      "Duodenectomy", "Ileectomy and ileostomy", "Incidental appendectomy",
      "Total abdominal colectomy with proctectomy and ileostomy",
      "Total abdominal colectomy with proctectomy and continent ileostomy",
      "Proctopexy combined with sigmoid resection by abdominal approach",
      "Excision of rectal procidentia with anastomosis by perineal approach",
      "Excision of lesion of large intestine", "Excision of lesion of small intestine",
      "Total resection of large intestine"
    )))),

    ibd = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Crohn's disease", "Crohn's disease of small intestine",
      "Crohn's disease of large bowel", "Crohn's disease of colon",
      "Crohn's disease of small AND large intestines",
      "Crohn's disease of terminal ileum", "Crohn's disease of ileum",
      "Crohn's disease of rectum", "Crohn's disease of duodenum",
      "Crohn's disease of intestine", "Crohn's disease in remission",
      "Gastrointestinal Crohn's disease", "Crohn disease of anal canal",
      "Ulcerative colitis", "Chronic ulcerative colitis",
      "Acute ulcerative colitis", "Chronic ulcerative pancolitis",
      "Chronic ulcerative enterocolitis", "Chronic ulcerative ileocolitis",
      "Chronic ulcerative rectosigmoiditis", "Chronic left-sided ulcerative colitis",
      "Left sided ulcerative colitis", "Ulcerative pancolitis",
      "Ulcerative proctocolitis", "Indeterminate colitis",
      "Microscopic colitis", "Collagenous colitis",
      "Inflammatory bowel disease", "Noninfectious colitis"
    )))),

    malabsorption = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Intestinal malabsorption", "Malabsorption syndrome",
      "Post-surgical malabsorption", "Short bowel syndrome",
      "Blind loop syndrome", "Celiac disease", "Adult form of celiac disease",
      "Sprue", "Tropical sprue", "Whipple's disease",
      "Exocrine pancreatic insufficiency", "Pancreatic insufficiency",
      "Secondary pancreatic insufficiency", "Pancreatic malabsorption",
      "Pancreatic steatorrhea", "Chronic steatorrhea",
      "Intestinal disaccharidase deficiency", "Sucrase-isomaltase deficiency",
      "Malabsorption syndrome due to intolerance to lactose",
      "Nonpersistence of intestinal lactase",
      "Bile acid malabsorption syndrome", "Bile acid malabsorption syndrome type III",
      "Postcholecystectomy diarrhea", "Protein-losing enteropathy",
      "Intestinal malabsorption of fat", "Malabsorption - iron",
      "Post-gastrointestinal tract surgery malnutrition",
      "Exocrine pancreatic manifestation co-occurrent and due to cystic fibrosis"
    )))),

    sarcoidosis = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Sarcoidosis", "Pulmonary sarcoidosis", "Cardiac sarcoidosis",
      "Neurosarcoidosis", "Cutaneous sarcoidosis", "Lymph node sarcoidosis",
      "Sarcoid heart muscle disease", "Sarcoid arthropathy", "Sarcoid arthritis",
      "Sarcoid myopathy", "Sarcoid neuropathy", "Sarcoid meningitis",
      "Subcutaneous sarcoidosis", "Sarcoidosis of digestive system",
      "Sarcoidosis of lung with sarcoidosis of lymph nodes",
      "Sarcoidosis, Darier-Roussy type", "Sarcoidosis, lupus pernio type",
      "Stage 3 pulmonary sarcoidosis", "Heerfordt's syndrome", "Lofgrens syndrome",
      "Demyelination of central nervous system co-occurrent and due to neurosarcoidosis",
      "Granulomatous sarcoid nephropathy"
    )))),

    spinal_injury = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Spinal cord injury", "Spinal cord injury without spinal bone injury",
      "Cervical spinal cord injury", "Thoracic cord injury without spinal bone injury",
      "Injury of cervical spinal cord", "Injury of thoracic spinal cord",
      "Injury of lumbar spinal cord", "Late effect of spinal cord injury",
      "Fracture of cervical spine", "Fracture of lumbar spine",
      "Fracture of thoracic spine", "Fracture of sacrum",
      "Closed fracture of cervical spine", "Closed fracture lumbar vertebra",
      "Closed fracture thoracic vertebra", "Traumatic spondylopathy",
      "Spinal dislocation with cervical cord lesion",
      "Spinal dislocation with thoracic cord lesion",
      "Spinal dislocation with lumbar cord lesion",
      "Complete lesion of cervical spinal cord", "Complete lesion of thoracic spinal cord",
      "Complete lesion of lumbar spinal cord"
    )))),

    cf = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Cystic fibrosis", "Cystic fibrosis of the lung",
      "Cystic fibrosis with meconium ileus", "Cystic fibrosis without meconium ileus",
      "Diabetes mellitus associated with cystic fibrosis",
      "Diabetes mellitus due to cystic fibrosis",
      "Pancreatic insufficiency due to cystic fibrosis of pancreas",
      "Exocrine pancreatic manifestation co-occurrent and due to cystic fibrosis"
    )))),

    pujo = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Congenital obstruction of ureteropelvic junction",
      "Congenital hydronephrosis due to ureteropelvic junction obstruction",
      "Acquired hydronephrosis due to ureteropelvic junction obstruction",
      "Obstruction of pelviureteric junction"
    )))),

    diverticulum = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Calcyceal diverticulum"
    )))),

    ureteric_stricture = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Stricture of ureter", "Acquired stricture of ureter",
      "Postoperative ureteric constriction"
    )))),

    vur = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Vesicoureteric reflux", "Vesicoureteral reflux without reflux nephropathy",
      "Congenital vesicoureterorenal reflux",
      "Secondary vesicoureteric reflux",
      "Secondary right vesicoureteral reflux grade 3",
      "Non-obstructive reflux-associated chronic pyelonephritis"
    )))),

    single_kidney = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Total nephrectomy", "Radical nephrectomy", "Recipient nephrectomy",
      "Transplant nephrectomy"
    )))),

    ileal_conduit = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Endoscopy of ileal conduit", "Ureteroileostomy"
    )))),

    eating_disorder = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Eating disorder", "Eating disorder in remission",
      "Anorexia nervosa", "Anorexia nervosa in remission",
      "Anorexia nervosa, restricting type", "Anorexia nervosa, binge-eating purging type",
      "Atypical anorexia nervosa", "Bulimia nervosa",
      "Bulimia nervosa in partial remission", "Bulimia nervosa, purging type",
      "Avoidant restrictive food intake disorder",
      "Rumination disorder", "Rumination disorder of infancy",
      "Adult rumination syndrome of ingested food",
      "Pica", "Pica of infancy and childhood", "Psychogenic overeating"
    )))),

    horseshoe_kidney = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
      "Horseshoe kidney"
    )))),

    .keep = "all"
  ) %>%
    mutate(
  
  .impaired_glucose = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
    "Diabetes mellitus without complication",
    "Diabetes mellitus",
    "Type 2 diabetes mellitus without complication",
    "Type 2 diabetes mellitus",
    "Gestational diabetes mellitus"
    
  )))),
  
  .central_obesity = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
    "Metabolic syndrome X",
      "High BMI",
      "Obesity"
    
  )))),
  
  .dyslipidaemia_metabolic = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
    "Dyslipidemia",
    "Complex dyslipidemia",
    "Dyslipidemia due to type 2 diabetes mellitus",
    "Dyslipidemia due to type 1 diabetes mellitus",
    "Dyslipidemia with high density lipoprotein below reference range and triglyceride above reference range due to type 2 diabetes mellitus"
  )))),
  
  .htn_metabolic = as.integer(map_lgl(standard_concept_names, ~ any(.x %in% c(
    "Essential hypertension", "Benign essential hypertension",
    "Hypertensive disorder", "Secondary hypertension"
  )))),
  
  # Metabolic disorder = 3 or more of the 4 components
  metabolic_disorder = as.integer(
    (.impaired_glucose + .central_obesity + .dyslipidaemia_metabolic + .htn_metabolic) >= 3
  ),
        .keep = "all"
) %>%
select(-starts_with("."))  %>%
select(-standard_concept_names)

In [ ]:
%%R

recurrent_uti_ids <- eau_risk %>%
  filter(person_id %in% km_recurrence$person_id) %>%
  left_join(
    km_recurrence %>% select(person_id, first_date),
    by = "person_id",
    relationship = "many-to-many"
  ) %>%
  filter(date_time < first_date) %>%
  filter(standard_concept_name %in% c(
    "Urinary tract infectious disease", "Acute urinary tract infection",
    "Chronic urinary tract infection", "Recurrent urinary tract infection",
    "Lower urinary tract infectious disease", "Upper urinary tract infection",
    "Acute upper urinary tract infection", "Acute lower urinary tract infection",
    "Febrile urinary tract infection", "Postoperative urinary tract infection",
    "Catheter-associated urinary tract infection",
    "Urinary tract infection in pregnancy",
    "Urinary tract infection caused by Escherichia coli",
    "Urinary tract infection caused by Enterococcus",
    "Urinary tract infection caused by Klebsiella",
    "Urinary tract infection caused by Pseudomonas",
    "Acute pyelonephritis", "Acute pyelonephritis without medullary necrosis",
    "Acute pyelonephritis with medullary necrosis",
    "Chronic pyelonephritis", "Chronic obstructive pyelonephritis",
    "Chronic pyelonephritis without medullary necrosis",
    "Chronic pyelonephritis with medullary necrosis",
    "Pyelonephritis", "Pyelonephritis in pregnancy",
    "Pyonephrosis", "Infective cystitis", "Infective urethritis",
    "Acute infective cystitis", "Recurrent infective cystitis",
    "Bacterial urinary infection", "Neonatal urinary tract infection",
    "Emphysematous pyelonephritis", "Emphysematous cystitis",
    "Infections of kidney in pregnancy", "Infections of bladder in pregnancy",
    "Infectious disorder of kidney",
    "Non-obstructive reflux-associated chronic pyelonephritis"
  )) %>%
  distinct(person_id, date_time) %>%   
  arrange(person_id, date_time) %>%
  mutate(date_time = anydate(date_time)) %>%
  group_by(person_id) %>%
  mutate(
    within_6mo = as.integer(
      as.numeric(lead(date_time) - date_time) <= 180
    ),
    within_1yr = as.integer(
      as.numeric(lead(date_time, 2) - date_time) <= 365
    )
  ) %>%
  summarise(
    recurrent_uti = as.integer(
      any(within_6mo == 1, na.rm = TRUE) |
      any(within_1yr == 1, na.rm = TRUE)
    )
  ) %>%
  ungroup()

# Then in pmhx_cleaned, replace the uti flag with a join
pmhx_cleaned <- pmhx_cleaned %>%
  left_join(recurrent_uti_ids, by = "person_id") %>%
  mutate(
    uti = as.integer(coalesce(recurrent_uti, 0L))
  ) %>%
  select(-recurrent_uti)

In [ ]:
%%R

str(pmhx_cleaned)

In [ ]:
%%R

n_distinct(km_recurrence$person_id)

In [ ]:
%%R

age_at_first_stone_df_2 <- km_recurrence %>%
    dplyr::select(person_id,
                 age_at_first_stone)

In [ ]:
%%R

high_risk <- km_recurrence %>%
    select(person_id) %>% 
    distinct(person_id) %>%
    left_join(
      pmhx_cleaned,
      by = "person_id") %>%
        mutate(metabolic_syndrome = case_when(
      dm + htn + dyslipidaemia > 2 ~ 1,
      dm + dyslipidaemia > 1 ~ 1,
      dyslipidaemia == 1 ~ 1,
      TRUE ~ 0
    )) %>%
        mutate(across(c(metabolic_disorder, metabolic_syndrome, hyperpth, uti, immunosuppression, 
                  transplant_immunosuppression, pkd, msk, 
                  bowel_surgery, ibd, malabsorption, sarcoidosis, spinal_injury, 
                  cf, pujo, diverticulum, ureteric_stricture, vur, single_kidney, 
                  ileal_conduit, eating_disorder, horseshoe_kidney),
                ~replace_na(., 0))) %>%
    mutate(
    high_risk = ifelse(
      (metabolic_disorder +
        metabolic_syndrome +
        hyperpth + immunosuppression + transplant_immunosuppression +
        pkd + msk + bowel_surgery + ibd + malabsorption +
        sarcoidosis + spinal_injury + cf + pujo + diverticulum + ureteric_stricture + 
        vur + single_kidney + ileal_conduit + eating_disorder + horseshoe_kidney)<1,
      "low_risk",
      "high_risk")) %>% 
    select(person_id,
           high_risk) %>%
    left_join(age_at_first_stone_df_2 %>% 
              dplyr::select(person_id, age_at_first_stone), 
             by = "person_id") %>%
    mutate(
        high_risk = case_when(
        age_at_first_stone < 25 ~ "high_risk",
        TRUE ~ high_risk)
    ) %>% 
    group_by(person_id) %>%
    summarise(
    high_risk = if_else(any(high_risk == "high_risk"), "high_risk", "low_risk"),
    .groups = "drop"
  )

table(high_risk$high_risk)

In [ ]:
%%R

str(high_risk)

In [ ]:
%%R
library(tidyverse)
n_distinct(high_risk$person_id)

## Table of Patients with High Risk characteristics

In [ ]:
%%R

pmhx_cleaned %>%
    left_join(age_at_first_stone_df %>% 
              dplyr::select(person_id, age_at_first_stone),
             by = "person_id") %>%
    mutate(metabolic_syndrome = case_when(
      dm + htn + dyslipidaemia > 2 ~ 1,
      dm + dyslipidaemia > 1 ~ 1,
      dyslipidaemia == 1 ~ 1,
      TRUE ~ 0
    ),
          young_age_of_onset = case_when(
              age_at_first_stone < 25 ~ 1,
              TRUE ~ 0
          )) %>%
  summarise(across(c(metabolic_disorder, metabolic_syndrome, hyperpth, immunosuppression,
                     transplant_immunosuppression, pkd, msk,
                     bowel_surgery, ibd, malabsorption, sarcoidosis, spinal_injury,
                     cf, pujo, diverticulum, ureteric_stricture, vur, single_kidney,
                     ileal_conduit, eating_disorder, horseshoe_kidney, young_age_of_onset),
                   ~ sum(. == 1, na.rm = TRUE))) %>%
  pivot_longer(everything(), names_to = "condition", values_to = "n") %>%
  mutate(pct = round(n / n_distinct(pmhx_cleaned$person_id) * 100, 1)) %>%
  arrange(desc(n)) %>%
  print(n=24)

In [ ]:
%%R

str(pmhx_cleaned)

In [ ]:
%%R
result <- km_recurrence %>%
  left_join(high_risk, by = "person_id") %>%
  drop_na(first_date) %>%
  group_by(person_id) %>%
  summarise(
    high_risk = if_else(any(high_risk == "high_risk"), "high_risk", "low_risk"),
    .groups = "drop"
  )


# Summary
cat("\nSummary:\n")
cat("Total patients:", nrow(result), "\n")
cat("High risk patients:", sum(result$high_risk == "high_risk"), "\n")
cat("Low risk patients:", sum(result$high_risk == "low_risk"), "\n")

## Compare with and without rUTIs

### Define datasets

In [ ]:
%%R

high_risk_with_uti <- km_recurrence %>%
    select(person_id) %>% 
    distinct(person_id) %>%
    left_join(
      pmhx_cleaned,
      by = "person_id") %>%
        mutate(metabolic_syndrome = case_when(
      dm + htn + dyslipidaemia > 2 ~ 1,
      dm + dyslipidaemia > 1 ~ 1,
      dyslipidaemia == 1 ~ 1,
      TRUE ~ 0
    )) %>%
        mutate(across(c(metabolic_disorder, metabolic_syndrome, hyperpth, uti, immunosuppression, 
                  transplant_immunosuppression, pkd, msk, 
                  bowel_surgery, ibd, malabsorption, sarcoidosis, spinal_injury, 
                  cf, pujo, diverticulum, ureteric_stricture, vur, single_kidney, 
                  ileal_conduit, eating_disorder, horseshoe_kidney),
                ~replace_na(., 0))) %>%
    mutate(
    high_risk = ifelse(
      (metabolic_disorder +
        metabolic_syndrome + uti +
        hyperpth + immunosuppression + transplant_immunosuppression +
        pkd + msk + bowel_surgery + ibd + malabsorption +
        sarcoidosis + spinal_injury + cf + pujo + diverticulum + ureteric_stricture + 
        vur + single_kidney + ileal_conduit + eating_disorder + horseshoe_kidney)<1,
      "low_risk",
      "high_risk")) %>% 
    select(person_id,
           high_risk) %>%
    left_join(age_at_first_stone_df_2 %>% 
              dplyr::select(person_id, age_at_first_stone), 
             by = "person_id") %>%
    mutate(
        high_risk = case_when(
        age_at_first_stone < 25 ~ "high_risk",
        TRUE ~ high_risk)
    ) %>% 
    group_by(person_id) %>%
    summarise(
    high_risk = if_else(any(high_risk == "high_risk"), "high_risk", "low_risk"),
    .groups = "drop"
  )


table(high_risk_with_uti$high_risk)

In [ ]:
%%R

high_risk_without_uti <- km_recurrence %>%
    select(person_id) %>% 
    distinct(person_id) %>%
    left_join(
      pmhx_cleaned,
      by = "person_id") %>%
        mutate(metabolic_syndrome = case_when(
      dm + htn + dyslipidaemia > 2 ~ 1,
      dm + dyslipidaemia > 1 ~ 1,
      dyslipidaemia == 1 ~ 1,
      TRUE ~ 0
    )) %>%
        mutate(across(c(metabolic_disorder, metabolic_syndrome, hyperpth, uti, immunosuppression, 
                  transplant_immunosuppression, pkd, msk, 
                  bowel_surgery, ibd, malabsorption, sarcoidosis, spinal_injury, 
                  cf, pujo, diverticulum, ureteric_stricture, vur, single_kidney, 
                  ileal_conduit, eating_disorder, horseshoe_kidney),
                ~replace_na(., 0))) %>%
    mutate(
    high_risk = ifelse(
      (metabolic_disorder +
        metabolic_syndrome + 
        hyperpth + immunosuppression + transplant_immunosuppression +
        pkd + msk + bowel_surgery + ibd + malabsorption +
        sarcoidosis + spinal_injury + cf + pujo + diverticulum + ureteric_stricture + 
        vur + single_kidney + ileal_conduit + eating_disorder + horseshoe_kidney)<1,
      "low_risk",
      "high_risk")) %>% 
    select(person_id,
           high_risk) %>%
    left_join(age_at_first_stone_df_2 %>% 
              dplyr::select(person_id, age_at_first_stone), 
             by = "person_id") %>%
    mutate(
        high_risk = case_when(
        age_at_first_stone < 25 ~ "high_risk",
        TRUE ~ high_risk)
    ) %>% 
    group_by(person_id) %>%
    summarise(
    high_risk = if_else(any(high_risk == "high_risk"), "high_risk", "low_risk"),
    .groups = "drop"
  )

table(high_risk_without_uti$high_risk)

#### With rUTI

In [ ]:
%%R
result_with_uti <- km_recurrence %>%
  left_join(high_risk_with_uti, by = "person_id") %>%
  drop_na(first_date) %>%
  group_by(person_id) %>%
  summarise(
    high_risk = if_else(any(high_risk == "high_risk"), "high_risk", "low_risk"),
    .groups = "drop"
  )


# Summary
cat("\nSummary:\n")
cat("Total patients:", nrow(result_with_uti), "\n")
cat("High risk patients:", sum(result_with_uti$high_risk == "high_risk"), "\n")
cat("Low risk patients:", sum(result_with_uti$high_risk == "low_risk"), "\n")

#### Without rUTI

In [ ]:
%%R
result_without_uti <- km_recurrence %>%
  left_join(high_risk_without_uti, by = "person_id") %>%
  drop_na(first_date) %>%
  group_by(person_id) %>%
  summarise(
    high_risk = if_else(any(high_risk == "high_risk"), "high_risk", "low_risk"),
    .groups = "drop"
  )


# Summary
cat("\nSummary:\n")
cat("Total patients:", nrow(result_without_uti), "\n")
cat("High risk patients:", sum(result_without_uti$high_risk == "high_risk"), "\n")
cat("Low risk patients:", sum(result_without_uti$high_risk == "low_risk"), "\n")

In [ ]:
%%R
overall_data_cut_off_5 <- km_recurrence2 %>%
  filter(
    !(pheno == 0 & (final_date - first_date)/365 < 5)
  )

str(overall_data_cut_off_5)

In [ ]:
%%R

table(overall_data_cut_off_5$recurrence)

In [ ]:
%%R

str(result_with_uti)

### Comparison of ROC plots

In [ ]:
%%R

# -------------------------------
# Prepare base dataset
# -------------------------------
base_data_with_uti <- overall_data_cut_off_5 %>%
    select(-high_risk) %>%
    left_join(result_with_uti, by = "person_id") %>%
    dplyr::select(person_id, time_to_event_yr, age_at_first_stone, recurrence, first_date, final_date, high_risk, initial_presentation) %>%
    mutate(
        count_diff = recurrence,
        last_date = final_date,
        .keep = "all"
    ) %>% select(c(
        person_id,
        count_diff,
        first_date,
        last_date,
        high_risk,
        initial_presentation,
        age_at_first_stone,
        recurrence
    ))


# -------------------------------
# Prepare km dataset
# -------------------------------
base_data_with_uti_for_km <- km_recurrence %>%
  left_join(high_risk_with_uti, by = "person_id") %>%
  mutate(recurrence = case_when(
                            recurrence == 0 ~ 0,
                            TRUE ~ 1
                                  ), 
         age_at_first_stone = as.numeric(age_at_first_stone),
         time_to_recurrence_1 = time_to_event_yr,
         .keep = "unused") %>%
        dplyr::select(person_id,
                     high_risk,
                     initial_presentation,
                     age_at_first_stone,
                     recurrence,
                     time_to_recurrence_1)



# -------------------------------
# Prepare base dataset
# -------------------------------
base_data_without_uti <- overall_data_cut_off_5 %>%
    select(-high_risk) %>%
    left_join(result_without_uti, by = "person_id") %>%
    dplyr::select(person_id, time_to_event_yr, age_at_first_stone, recurrence, first_date, final_date, high_risk, initial_presentation) %>%
    mutate(
        count_diff = recurrence,
        last_date = final_date,
        .keep = "all"
    ) %>% select(c(
        person_id,
        count_diff,
        first_date,
        last_date,
        high_risk,
        initial_presentation,
        age_at_first_stone,
        recurrence
    ))


# -------------------------------
# Prepare km dataset
# -------------------------------
base_data_without_uti_for_km <- km_recurrence %>%
  left_join(high_risk_without_uti, by = "person_id") %>%
  mutate(recurrence = case_when(
                            recurrence == 0 ~ 0,
                            TRUE ~ 1
                                  ), 
         age_at_first_stone = as.numeric(age_at_first_stone),
         time_to_recurrence_1 = time_to_event_yr,
         .keep = "unused") %>%
        dplyr::select(person_id,
                     high_risk,
                     initial_presentation,
                     age_at_first_stone,
                     recurrence,
                     time_to_recurrence_1)

In [ ]:
%%R

table(base_data_with_uti$high_risk)

In [ ]:
%%R

# -------------------------------
# Build combined results table
# -------------------------------
results_df <- data.frame(
  Dataset = character(),
  Comparison = character(),
  AUC_CI = character(),
  Sensitivity = character(),
  Specificity = character(),
  DeLong_p = character(),
  stringsAsFactors = FALSE
)

for (i in seq_along(thresholds)) {
  thresh <- thresholds[i]
  
  prep_roc <- function(base_data) {
    base_data %>%
      mutate(
        actual = case_when(
          count_diff >= thresh ~ 1,
          count_diff == 0 ~ 0,
          TRUE ~ NA_integer_
        ),
        predicted = ifelse(high_risk == "high_risk", 1, 0)
      ) %>%
      drop_na()
  }
  
  roc_with    <- prep_roc(base_data_with_uti)
  roc_without <- prep_roc(base_data_without_uti)
  
  roc_obj1 <- roc(roc_with$actual, roc_with$predicted, quiet = TRUE)
  roc_obj2 <- roc(roc_without$actual, roc_without$predicted, quiet = TRUE)
  
  # DeLong test
  test_result <- roc.test(roc_obj1, roc_obj2, method = "delong")
  p_val <- test_result$p.value
  p_label <- ifelse(p_val < 0.001, "<0.001", sprintf("%.3f", p_val))
  
  # Extract metrics for each dataset
  extract_metrics <- function(roc_obj, dataset_name, label, p_text) {
    auc_val <- as.numeric(auc(roc_obj))
    auc_ci  <- ci.auc(roc_obj)
    
    # Best threshold coords (Youden index)
    coords_best <- coords(roc_obj, "best", ret = c("sensitivity", "specificity"),
                          best.method = "youden")
    
    # CIs for sensitivity and specificity at best threshold
    sens_ci <- ci.se(roc_obj, specificities = coords_best$specificity)
    spec_ci <- ci.sp(roc_obj, sensitivities = coords_best$sensitivity)
    
    data.frame(
      Dataset = dataset_name,
      Comparison = label,
      AUC_CI = sprintf("%.2f (%.2f–%.2f)", auc_val, auc_ci[1], auc_ci[3]),
      Sensitivity = sprintf("%.2f (%.2f–%.2f)",
                            coords_best$sensitivity, sens_ci[, 1], sens_ci[, 3]),
      Specificity = sprintf("%.2f (%.2f–%.2f)",
                            coords_best$specificity, spec_ci[, 1], spec_ci[, 3]),
      DeLong_p = p_text
    )
  }
  
  results_df <- rbind(
    results_df,
    extract_metrics(roc_obj1, "With UTI", labels[i], p_label),
    extract_metrics(roc_obj2, "Without UTI", labels[i], "")
  )
}

results_df

## Generate Plots for Publication

In [ ]:
%%R

# -------------------------------
# Prepare base dataset
# -------------------------------
base_data_with_uti <- overall_data_cut_off_5 %>%
    select(-high_risk) %>%
    left_join(result_with_uti, by = "person_id") %>%
    dplyr::select(person_id, time_to_event_yr, age_at_first_stone, recurrence, first_date, final_date, high_risk, initial_presentation) %>%
    mutate(
        count_diff = recurrence,
        last_date = final_date,
        .keep = "all"
    ) %>% select(c(
        person_id,
        count_diff,
        first_date,
        last_date,
        high_risk,
        initial_presentation,
        age_at_first_stone,
        recurrence
    ))


# -------------------------------
# Prepare km dataset
# -------------------------------
base_data_with_uti_for_km <- km_recurrence %>%
  left_join(high_risk_with_uti, by = "person_id") %>%
  mutate(recurrence = case_when(
                            recurrence == 0 ~ 0,
                            TRUE ~ 1
                                  ), 
         age_at_first_stone = as.numeric(age_at_first_stone),
         time_to_recurrence_1 = time_to_event_yr,
         .keep = "unused") %>%
        dplyr::select(person_id,
                     high_risk,
                     initial_presentation,
                     age_at_first_stone,
                     recurrence,
                     time_to_recurrence_1)



# -------------------------------
# Prepare base dataset
# -------------------------------
base_data_without_uti <- overall_data_cut_off_5 %>%
    select(-high_risk) %>%
    left_join(result_without_uti, by = "person_id") %>%
    dplyr::select(person_id, time_to_event_yr, age_at_first_stone, recurrence, first_date, final_date, high_risk, initial_presentation) %>%
    mutate(
        count_diff = recurrence,
        last_date = final_date,
        .keep = "all"
    ) %>% select(c(
        person_id,
        count_diff,
        first_date,
        last_date,
        high_risk,
        initial_presentation,
        age_at_first_stone,
        recurrence
    ))


# -------------------------------
# Prepare km dataset
# -------------------------------
base_data_without_uti_for_km <- km_recurrence %>%
  left_join(high_risk_without_uti, by = "person_id") %>%
  mutate(recurrence = case_when(
                            recurrence == 0 ~ 0,
                            TRUE ~ 1
                                  ), 
         age_at_first_stone = as.numeric(age_at_first_stone),
         time_to_recurrence_1 = time_to_event_yr,
         .keep = "unused") %>%
        dplyr::select(person_id,
                     high_risk,
                     initial_presentation,
                     age_at_first_stone,
                     recurrence,
                     time_to_recurrence_1)

In [ ]:
%%R

## Generic parameters ####
thresholds <- c(1, 2, 3)
labels <- c("0 vs ≥1 events", "0 vs ≥2 events", "0 vs ≥3 events")
colors <- c("#1b9e77", "#d95f02", "#7570b3")

## Generate matched datasets ####
### Matched without UTI ####
base_data_match_without_uti <- base_data_without_uti %>%
  mutate(
    high_risk_binary = if_else(high_risk == "high_risk", 1L, 0L),
    age_at_first_stone = as.numeric(age_at_first_stone),
    initial_presentation = as.factor(initial_presentation)
  ) %>%
  drop_na(high_risk_binary, age_at_first_stone, initial_presentation)

m <- matchit(high_risk_binary ~ age_at_first_stone + initial_presentation,
             data = base_data_match_without_uti,
             method = "nearest")

base_data_match_without_uti <- match.data(m)

### Matched with UTI ####
base_data_match_with_uti <- base_data_with_uti %>%
  mutate(
    high_risk_binary = if_else(high_risk == "high_risk", 1L, 0L),
    age_at_first_stone = as.numeric(age_at_first_stone),
    initial_presentation = as.factor(initial_presentation)
  ) %>%
  drop_na(high_risk_binary, age_at_first_stone, initial_presentation)

m <- matchit(high_risk_binary ~ age_at_first_stone + initial_presentation,
             data = base_data_match_with_uti,
             method = "nearest")

base_data_match_with_uti <- match.data(m)

## Helper functions ####
prep_roc <- function(base_data, thresh) {
  base_data %>%
    mutate(
      actual = case_when(
        count_diff >= thresh ~ 1,
        count_diff == 0 ~ 0,
        TRUE ~ NA_integer_
      ),
      predicted = if_else(high_risk == "high_risk", 1, 0)
    ) %>%
    drop_na()
}

extract_metrics <- function(roc_obj, dataset_name, label, p_text) {
  auc_val <- as.numeric(auc(roc_obj))
  auc_ci  <- ci.auc(roc_obj)

  coords_best <- coords(roc_obj, "best", ret = c("sensitivity", "specificity"),
                        best.method = "youden")

  sens_ci <- ci.se(roc_obj, specificities = coords_best$specificity)
  spec_ci <- ci.sp(roc_obj, sensitivities = coords_best$sensitivity)

  data.frame(
    Dataset = dataset_name,
    Comparison = label,
    AUC_CI = sprintf("%.2f (%.2f–%.2f)", auc_val, auc_ci[1], auc_ci[3]),
    Sensitivity = sprintf("%.2f (%.2f–%.2f)",
                          coords_best$sensitivity, sens_ci[, 1], sens_ci[, 3]),
    Specificity = sprintf("%.2f (%.2f–%.2f)",
                          coords_best$specificity, spec_ci[, 1], spec_ci[, 3]),
    DeLong_p = p_text
  )
}

## Compare ROC: DeLong's tests ####
### Unmatched: Without UTI vs With UTI ####
results_unmatched_df <- data.frame(
  Dataset = character(),
  Comparison = character(),
  AUC_CI = character(),
  Sensitivity = character(),
  Specificity = character(),
  DeLong_p = character(),
  stringsAsFactors = FALSE
)

for (i in seq_along(thresholds)) {
  thresh <- thresholds[i]

  roc_with    <- prep_roc(base_data_with_uti, thresh)
  roc_without <- prep_roc(base_data_without_uti, thresh)

  roc_obj1 <- roc(roc_with$actual, roc_with$predicted, quiet = TRUE)
  roc_obj2 <- roc(roc_without$actual, roc_without$predicted, quiet = TRUE)

  test_result <- roc.test(roc_obj1, roc_obj2, method = "delong")
  p_val <- test_result$p.value
  p_label <- if_else(p_val < 0.001, "<0.001", sprintf("%.3f", p_val))

  results_unmatched_df <- rbind(
    results_unmatched_df,
    extract_metrics(roc_obj1, "With UTI", labels[i], p_label),
    extract_metrics(roc_obj2, "Without UTI", labels[i], "")
  )
}

results_unmatched_df %>%
  gt() %>%
  tab_header(title = "Unmatched: Without UTI vs With UTI") %>%
  gt_theme_espn()

### Matched: Without UTI vs With UTI ####
results_matched_df <- data.frame(
  Dataset = character(),
  Comparison = character(),
  AUC_CI = character(),
  Sensitivity = character(),
  Specificity = character(),
  DeLong_p = character(),
  stringsAsFactors = FALSE
)

for (i in seq_along(thresholds)) {
  thresh <- thresholds[i]

  roc_with    <- prep_roc(base_data_match_with_uti, thresh)
  roc_without <- prep_roc(base_data_match_without_uti, thresh)

  roc_obj1 <- roc(roc_with$actual, roc_with$predicted, quiet = TRUE)
  roc_obj2 <- roc(roc_without$actual, roc_without$predicted, quiet = TRUE)

  test_result <- roc.test(roc_obj1, roc_obj2, method = "delong")
  p_val <- test_result$p.value
  p_label <- if_else(p_val < 0.001, "<0.001", sprintf("%.3f", p_val))

  results_matched_df <- rbind(
    results_matched_df,
    extract_metrics(roc_obj1, "With UTI", labels[i], p_label),
    extract_metrics(roc_obj2, "Without UTI", labels[i], "")
  )
}

results_matched_df %>%
  gt() %>%
  tab_header(title = "Matched: Without UTI vs With UTI") %>%
  gt_theme_espn()

## ROC Plots ####
### Helper function for ROC plot generation ####
generate_roc_plot <- function(base_data, plot_title) {
  roc_list <- list()
  auc_table_df <- data.frame(Comparison = character(), AUC_CI = character())
  ci_combined_df <- data.frame()

  for (i in seq_along(thresholds)) {
    thresh <- thresholds[i]
    label <- labels[i]

    roc_data <- prep_roc(base_data, thresh)
    roc_obj <- roc(roc_data$actual, roc_data$predicted, quiet = TRUE)

    roc_list[[i]] <- list(roc_obj = roc_obj, color = colors[i], label = label)

    auc_val <- as.numeric(auc(roc_obj))
    auc_ci  <- ci.auc(roc_obj)

    auc_table_df <- rbind(
      auc_table_df,
      data.frame(
        Comparison = label,
        AUC_CI = sprintf("%.2f (%.2f–%.2f)", auc_val, auc_ci[1], auc_ci[3])
      )
    )

    ci_se <- ci.se(roc_obj, specificities = seq(0, 1, length.out = 100))
    ci_combined_df <- rbind(ci_combined_df, data.frame(
      fpr = 1 - as.numeric(rownames(ci_se)),
      tpr_lower = ci_se[, 1],
      tpr_upper = ci_se[, 3],
      Comparison = label
    ))
  }

  roc_combined_df <- do.call(rbind, lapply(seq_along(roc_list), function(i) {
    roc_obj <- roc_list[[i]]$roc_obj
    data.frame(
      fpr = 1 - roc_obj$specificities,
      tpr = roc_obj$sensitivities,
      Comparison = roc_list[[i]]$label
    )
  }))

  auc_table <- tableGrob(auc_table_df, rows = NULL, theme = ttheme_minimal(base_size = 9))
  auc_table_boxed <- grobTree(
    rectGrob(gp = gpar(col = "black", fill = "white", lwd = 1)),
    auc_table
  )

  ggplot() +
    geom_ribbon(
      data = ci_combined_df,
      aes(x = fpr, ymin = tpr_lower, ymax = tpr_upper, fill = Comparison),
      alpha = 0.2
    ) +
    geom_line(
      data = roc_combined_df,
      aes(x = fpr, y = tpr, color = Comparison),
      linewidth = 1
    ) +
    geom_abline(linetype = "dashed", alpha = 0.6) +
    labs(x = "False Positive Rate", y = "True Positive Rate", title = plot_title) +
    scale_color_manual(values = colors) +
    scale_fill_manual(values = colors) +
    coord_equal() +
    theme_minimal(base_size = 12) +
    annotation_custom(
      grob = auc_table_boxed,
      xmin = 0.55, xmax = 1,
      ymin = 0.05, ymax = 0.3
    ) + theme_bw()
}

# Without UTI ####
### Unmatched ####
roc_plot_without_uti_unmatched <- generate_roc_plot(base_data_without_uti, "ROC Curves: EAU Risk Stratification")
roc_plot_without_uti_unmatched

### Matched ####
roc_plot_without_uti_matched <- generate_roc_plot(base_data_match_without_uti, "ROC Curves: EAU Risk Stratification")
roc_plot_without_uti_matched

### Split by Initial Presentation ####
presentations <- unique(base_data_without_uti$initial_presentation)

roc_combined_df <- data.frame()
ci_combined_df  <- data.frame()
auc_table_df    <- data.frame()

for (pres in presentations) {
  pres_data <- base_data_without_uti %>% filter(initial_presentation == pres)
  
  for (i in seq_along(thresholds)) {
    thresh <- thresholds[i]
    label  <- labels[i]
    
    roc_data <- pres_data %>%
      mutate(
        actual = case_when(
          count_diff >= thresh ~ 1,
          count_diff == 0      ~ 0,
          TRUE                 ~ NA_integer_
        ),
        predicted = if_else(high_risk == "high_risk", 1, 0)
      ) %>%
      drop_na()
    
    # Skip if insufficient data
    if (nrow(roc_data) < 10 || length(unique(roc_data$actual)) < 2) next
    
    roc_obj <- roc(roc_data$actual, roc_data$predicted, quiet = TRUE)
    
    # ROC line
    roc_combined_df <- rbind(roc_combined_df, data.frame(
      fpr                  = 1 - roc_obj$specificities,
      tpr                  = roc_obj$sensitivities,
      Comparison           = label,
      initial_presentation = pres
    ))
    
    # CI ribbon
    ci_se <- ci.se(roc_obj, specificities = seq(0, 1, length.out = 100))
    ci_combined_df <- rbind(ci_combined_df, data.frame(
      fpr                  = 1 - as.numeric(rownames(ci_se)),
      tpr_lower            = ci_se[, 1],
      tpr_upper            = ci_se[, 3],
      Comparison           = label,
      initial_presentation = pres
    ))
    
    # AUC + CI
    auc_val <- as.numeric(auc(roc_obj))
    auc_ci  <- ci.auc(roc_obj)
    auc_table_df <- rbind(auc_table_df, data.frame(
      initial_presentation = pres,
      Comparison           = label,
      AUC                  = sprintf("%.2f (%.2f–%.2f)", auc_val, auc_ci[1], auc_ci[3])
    ))
  }
}


roc_plot_facet_by_any_procedure_without_uti <- ggplot() +
  geom_ribbon(
    data = ci_combined_df,
    aes(x = fpr, ymin = tpr_lower, ymax = tpr_upper, fill = Comparison),
    alpha = 0.2
  ) +
  geom_line(
    data = roc_combined_df,
    aes(x = fpr, y = tpr, color = Comparison),
    linewidth = 1
  ) +
  geom_abline(linetype = "dashed", alpha = 0.6) +
  # AUC labels in each facet
  geom_label(
    data = auc_table_df,
    aes(label = paste0(Comparison, ": ", AUC), color = Comparison),
    x = 0.95, y = seq(0.25, 0.05, length.out = length(labels))[
      match(auc_table_df$Comparison, labels)
    ],
    hjust = 1, size = 2.8, fill = "white", label.size = 0.3, show.legend = FALSE
  ) +
  scale_color_manual(values = setNames(colors, labels)) +
  scale_fill_manual(values  = setNames(colors, labels)) +
  facet_wrap(~ initial_presentation) +
  coord_equal() +
  labs(
    x     = "False Positive Rate",
    y     = "True Positive Rate",
    title = "ROC Curves: EAU Risk Stratification by Initial Presentation",
    color = NULL, fill = NULL
  ) +
  theme_bw(base_size = 11) +
  theme(
    legend.position = "bottom",
    strip.text      = element_text(face = "bold")
  )

roc_plot_facet_by_any_procedure_without_uti

## Create KM Plots ####
base_data_without_uti_for_km1 <- base_data_without_uti_for_km %>%
  mutate(
    high_risk_binary = if_else(high_risk == "high_risk", 1L, 0L),
    age_at_first_stone = as.numeric(age_at_first_stone),
    initial_presentation = as.factor(initial_presentation)
  ) %>%
  drop_na(high_risk_binary, age_at_first_stone, initial_presentation)

m <- matchit(high_risk_binary ~ age_at_first_stone + initial_presentation,
             data = base_data_without_uti_for_km1,
             method = "nearest")

base_data_match_without_uti_match <- match.data(m)


### Unmatched ####
km_time_to_event_colic_to_colic_risk_status_fit <-
  surv_fit(Surv(time_to_recurrence_1, recurrence) ~ high_risk,
           data =
             base_data_without_uti_for_km)

first_to_second_event_by_risk_without_uti_unmatched <- ggsurvplot(
  km_time_to_event_colic_to_colic_risk_status_fit,
  data = base_data_without_uti_for_km,
  censor = FALSE,
  risk.table = TRUE,
  surv.median.line = "hv",
  conf.int = TRUE,
  tables.theme = theme_cleantable(),
  break.time.by = 1,
  xlab = "Time (years)",
  xlim = c(0, 10),
  title = "Time to First Recurrence",
  pval = TRUE,
  legend.title = "Risk Status",
  legend.labs = c("High Risk", "Low Risk"),          
  ggtheme = theme_bw()
)

first_to_second_event_by_risk_without_uti_unmatched
 
### Matched ####
# Define variables
km_time_to_event_colic_to_colic_risk_status_fit <-
  surv_fit(Surv(time_to_recurrence_1, recurrence) ~ high_risk,
           data =
             base_data_match_without_uti_match)

first_to_second_event_by_risk_without_uti_matched <- ggsurvplot(
  km_time_to_event_colic_to_colic_risk_status_fit,
  data = base_data_match_without_uti_match,
  censor = FALSE,
  risk.table = TRUE,
  surv.median.line = "hv",
  conf.int = TRUE,
  tables.theme = theme_cleantable(),
  break.time.by = 1,
  xlab = "Time (years)",
  xlim = c(0, 10),
  title = "Time to First Recurrence",
  pval = TRUE,
  legend.title = "Risk Status",
  legend.labs = c("High Risk", "Low Risk"),          
  ggtheme = theme_bw()
)

first_to_second_event_by_risk_without_uti_matched

### Split by Initial Presentation ####
km_fit <- surv_fit(
  Surv(time_to_recurrence_1, recurrence) ~ high_risk,
  data = as.data.frame(base_data_without_uti_for_km)
)

first_to_second_event_by_risk_facet_without_uti <- ggsurvplot_facet(
  km_fit,
  data             = as.data.frame(base_data_without_uti_for_km),
  facet.by         = "initial_presentation",
  censor           = FALSE,
  conf.int         = TRUE,
  surv.median.line = "hv",
  pval             = TRUE,
  break.time.by    = 1,
  xlim             = c(0, 10),
  xlab             = "Time (years)",
  ylab             = "Recurrence-free probability",
  legend.title     = "Risk Status",
  legend.labs      = c("High Risk", "Low Risk"),
  ggtheme          = theme_bw(),
  panel.labs.font  = list(face = "bold")
)

first_to_second_event_by_risk_facet_without_uti

# With UTI ####
## Create ROC plots ####
### Unmatched ####
roc_plot_with_uti_unmatched <- generate_roc_plot(base_data_with_uti, "ROC Curves: EAU Risk Stratification")
roc_plot_with_uti_unmatched

### Matched ####
roc_plot_with_uti_matched <- generate_roc_plot(base_data_match_with_uti, "ROC Curves: EAU Risk Stratification")
roc_plot_with_uti_matched

### Split by Initial Presentation ####
presentations <- unique(base_data_with_uti$initial_presentation)

roc_combined_df <- data.frame()
ci_combined_df  <- data.frame()
auc_table_df    <- data.frame()

for (pres in presentations) {
  pres_data <- base_data_with_uti %>% filter(initial_presentation == pres)
  
  for (i in seq_along(thresholds)) {
    thresh <- thresholds[i]
    label  <- labels[i]
    
    roc_data <- pres_data %>%
      mutate(
        actual = case_when(
          count_diff >= thresh ~ 1,
          count_diff == 0      ~ 0,
          TRUE                 ~ NA_integer_
        ),
        predicted = if_else(high_risk == "high_risk", 1, 0)
      ) %>%
      drop_na()
    
    # Skip if insufficient data
    if (nrow(roc_data) < 10 || length(unique(roc_data$actual)) < 2) next
    
    roc_obj <- roc(roc_data$actual, roc_data$predicted, quiet = TRUE)
    
    # ROC line
    roc_combined_df <- rbind(roc_combined_df, data.frame(
      fpr                  = 1 - roc_obj$specificities,
      tpr                  = roc_obj$sensitivities,
      Comparison           = label,
      initial_presentation = pres
    ))
    
    # CI ribbon
    ci_se <- ci.se(roc_obj, specificities = seq(0, 1, length.out = 100))
    ci_combined_df <- rbind(ci_combined_df, data.frame(
      fpr                  = 1 - as.numeric(rownames(ci_se)),
      tpr_lower            = ci_se[, 1],
      tpr_upper            = ci_se[, 3],
      Comparison           = label,
      initial_presentation = pres
    ))
    
    # AUC + CI
    auc_val <- as.numeric(auc(roc_obj))
    auc_ci  <- ci.auc(roc_obj)
    auc_table_df <- rbind(auc_table_df, data.frame(
      initial_presentation = pres,
      Comparison           = label,
      AUC                  = sprintf("%.2f (%.2f–%.2f)", auc_val, auc_ci[1], auc_ci[3])
    ))
  }
}


roc_plot_facet_by_any_procedure_with_uti <- ggplot() +
  geom_ribbon(
    data = ci_combined_df,
    aes(x = fpr, ymin = tpr_lower, ymax = tpr_upper, fill = Comparison),
    alpha = 0.2
  ) +
  geom_line(
    data = roc_combined_df,
    aes(x = fpr, y = tpr, color = Comparison),
    linewidth = 1
  ) +
  geom_abline(linetype = "dashed", alpha = 0.6) +
  # AUC labels in each facet
  geom_label(
    data = auc_table_df,
    aes(label = paste0(Comparison, ": ", AUC), color = Comparison),
    x = 0.95, y = seq(0.25, 0.05, length.out = length(labels))[
      match(auc_table_df$Comparison, labels)
    ],
    hjust = 1, size = 2.8, fill = "white", label.size = 0.3, show.legend = FALSE
  ) +
  scale_color_manual(values = setNames(colors, labels)) +
  scale_fill_manual(values  = setNames(colors, labels)) +
  facet_wrap(~ initial_presentation) +
  coord_equal() +
  labs(
    x     = "False Positive Rate",
    y     = "True Positive Rate",
    title = "ROC Curves: EAU Risk Stratification by Initial Presentation",
    color = NULL, fill = NULL
  ) +
  theme_bw(base_size = 11) +
  theme(
    legend.position = "bottom",
    strip.text      = element_text(face = "bold")
  )

roc_plot_facet_by_any_procedure_with_uti

## Create KM Plots ####
base_data_with_uti_for_km1 <- base_data_with_uti_for_km %>%
  mutate(
    high_risk_binary = if_else(high_risk == "high_risk", 1L, 0L),
    age_at_first_stone = as.numeric(age_at_first_stone),
    initial_presentation = as.factor(initial_presentation)
  ) %>%
  drop_na(high_risk_binary, age_at_first_stone, initial_presentation)

m <- matchit(high_risk_binary ~ age_at_first_stone + initial_presentation,
             data = base_data_with_uti_for_km1,
             method = "nearest")

base_data_match_with_uti_match <- match.data(m)

### Unmatched ####
# Define variables
km_time_to_event_colic_to_colic_risk_status_fit <-
  surv_fit(Surv(time_to_recurrence_1, recurrence) ~ high_risk,
           data =
             base_data_with_uti_for_km)

first_to_second_event_by_risk_with_uti_unmatched <- ggsurvplot(
  km_time_to_event_colic_to_colic_risk_status_fit,
  data = base_data_with_uti_for_km,
  censor = FALSE,
  risk.table = TRUE,
  surv.median.line = "hv",
  conf.int = TRUE,
  tables.theme = theme_cleantable(),
  break.time.by = 1,
  xlab = "Time (years)",
  xlim = c(0, 10),
  title = "Time to First Recurrence",
  pval = TRUE,
  legend.title = "Risk Status",
  legend.labs = c("High Risk", "Low Risk"),          
  ggtheme = theme_bw()
)

first_to_second_event_by_risk_with_uti_unmatched

### Matched ####
km_time_to_event_colic_to_colic_risk_status_fit <-
  surv_fit(Surv(time_to_recurrence_1, recurrence) ~ high_risk,
           data =
             base_data_match_with_uti_match)

first_to_second_event_by_risk_with_uti_matched <- ggsurvplot(
  km_time_to_event_colic_to_colic_risk_status_fit,
  data = base_data_match_with_uti_match,
  censor = FALSE,
  risk.table = TRUE,
  surv.median.line = "hv",
  conf.int = TRUE,
  tables.theme = theme_cleantable(),
  break.time.by = 1,
  xlab = "Time (years)",
  xlim = c(0, 10),
  title = "Time to First Recurrence",
  pval = TRUE,
  legend.title = "Risk Status",
  legend.labs = c("High Risk", "Low Risk"),          
  ggtheme = theme_bw()
)

first_to_second_event_by_risk_with_uti_matched

### Split by Initial Presentation ####
km_fit <- surv_fit(
  Surv(time_to_recurrence_1, recurrence) ~ high_risk,
  data = as.data.frame(base_data_with_uti_for_km)
)

first_to_second_event_by_risk_facet_with_uti <- ggsurvplot_facet(
  km_fit,
  data             = as.data.frame(base_data_with_uti_for_km),
  facet.by         = "initial_presentation",
  censor           = FALSE,
  conf.int         = TRUE,
  surv.median.line = "hv",
  pval             = TRUE,
  break.time.by    = 1,
  xlim             = c(0, 10),
  xlab             = "Time (years)",
  ylab             = "Recurrence-free probability",
  legend.title     = "Risk Status",
  legend.labs      = c("High Risk", "Low Risk"),
  ggtheme          = theme_bw(),
  panel.labs.font  = list(face = "bold")
)

first_to_second_event_by_risk_facet_with_uti

### Without UTI

In [ ]:
%%R

table(base_data_without_uti_for_km$high_risk)

In [ ]:
%%R
prepared_data <- base_data_without_uti_for_km %>%
  left_join(
      pmhx_cleaned,
      by = "person_id") %>%
        mutate(metabolic_syndrome = case_when(
      dm + htn + dyslipidaemia > 2 ~ 1,
      dm + dyslipidaemia > 1 ~ 1,
      dyslipidaemia == 1 ~ 1,
      TRUE ~ 0
    )) %>%
  mutate(
    early_onset = if_else(age_at_first_stone < 25, "1", "0")
  ) %>%
  select(-c(time_to_recurrence_1, person_id, first_date, dm, htn)) %>%
  mutate(across(1:27, ~ replace_na(as.character(.), "0"))) 

# Continuous summary
continuous_summary <- prepared_data %>%
  summarise(
    variable = "age_at_first_stone",
    summary = paste0(round(mean(as.numeric(age_at_first_stone), na.rm = TRUE), 1), 
                     " ± ", 
                     round(sd(as.numeric(age_at_first_stone), na.rm = TRUE), 1))
  )

# Categorical summary
categorical_summary <- prepared_data %>%
  select(-age_at_first_stone) %>%
  pivot_longer(everything(), names_to = "variable", values_to = "value") %>%
  count(variable, value) %>%
  group_by(variable) %>%
  mutate(
    total = sum(n),
    summary = paste0(n, " (", round(n / total * 100, 1), "%)")
  ) %>%
  ungroup() %>%
  filter(value == "1") %>%
  arrange(desc(n)) %>%
  select(variable, summary)

# Combine
full_summary <- bind_rows(continuous_summary, categorical_summary)
full_summary %>% print(n=59)

### With UTI

In [ ]:
%%R

table(base_data_with_uti_for_km$high_risk)

### deLong's test results

In [ ]:
%%R

results_unmatched_df 

In [ ]:
%%R

results_matched_df

## Save Plots

### Ensure plots ok

In [ ]:
%%R
roc_plot_with_uti_unmatched

In [ ]:
%%R
roc_plot_without_uti_unmatched

In [ ]:
%%R
first_to_second_event_by_risk_facet_without_uti

In [ ]:
%%R
first_to_second_event_by_risk_facet_with_uti

In [ ]:
%%R
roc_plot_with_uti_matched

In [ ]:
%%R
roc_plot_without_uti_matched

In [ ]:
%%R
first_to_second_event_by_risk_with_uti_unmatched

In [ ]:
%%R
first_to_second_event_by_risk_without_uti_unmatched

In [ ]:
%%R
first_to_second_event_by_risk_with_uti_matched

In [ ]:
%%R
first_to_second_event_by_risk_without_uti_matched

In [ ]:
%%R
first_to_second_event_by_risk_facet_with_uti

In [ ]:
%%R
first_to_second_event_by_risk_facet_without_uti

### Save plots

In [ ]:
%%R
ggsave("roc_plot_without_uti_aou.pdf", plot = roc_plot_without_uti_unmatched, 
       width = 8, height = 6)
ggsave("roc_plot_with_uti_aou.pdf", plot = roc_plot_with_uti_unmatched, 
       width = 8, height = 6)

ggsave("roc_plot_matched_without_uti_aou.pdf", plot = roc_plot_without_uti_matched, 
       width = 8, height = 6)
ggsave("roc_plot_matched_with_uti_aou.pdf", plot = roc_plot_with_uti_matched, 
       width = 8, height = 6)

ggsave("roc_plot_facet_by_any_procedure_with_uti_aou.pdf", 
       plot = roc_plot_facet_by_any_procedure_without_uti, 
       width = 10, height = 8)
ggsave("roc_plot_facet_by_any_procedure_without_uti_aou.pdf", 
       plot = roc_plot_facet_by_any_procedure_with_uti, 
       width = 10, height = 8)

In [ ]:
%%R

pdf("km_plot_unmatched_without_uti_aou.pdf", width = 10, height = 7)
print(first_to_second_event_by_risk_without_uti_unmatched)
dev.off()

pdf("km_plot_unmatched_with_uti_aou.pdf", width = 10, height = 7)
print(first_to_second_event_by_risk_with_uti_unmatched)
dev.off()

pdf("km_plot_matched_without_uti_aou.pdf", width = 10, height = 7)
print(first_to_second_event_by_risk_without_uti_matched)
dev.off()

pdf("km_plot_matched_with_uti_aou.pdf", width = 10, height = 7)
print(first_to_second_event_by_risk_with_uti_matched)
dev.off()

pdf("km_plot_facet_without_uti_aou.pdf", width=10, height=7)
print(first_to_second_event_by_risk_facet_without_uti)
dev.off()

pdf("km_plot_facet_with_uti_aou.pdf", width=10, height=7)
print(first_to_second_event_by_risk_facet_with_uti)
dev.off()